In [4]:
import pandas as pd
import os

def remove_spaces(raw_keywords):
    """
    한 행의 키워드 데이터를 받아 쉼표 기준으로 분리한 뒤,
    각 키워드 내부의 공백(띄어쓰기)을 모두 제거하여 덩어리 키워드로 만듭니다.
    """
    # 값이 비어있을 경우(NaN) 그대로 반환
    if pd.isna(raw_keywords):
        return raw_keywords
    
    final_words = []
    # 1. 쉼표(,)로 분리
    comma_split = str(raw_keywords).split(',')
    
    for item in comma_split:
        # 2. 키워드 내의 모든 공백(띄어쓰기) 제거
        # 예: '인테리어 소품' -> '인테리어소품'
        word_no_space = item.replace(" ", "")
        
        # 공백만 있던 문자열이 아니었다면 리스트에 추가
        if word_no_space: 
            final_words.append(word_no_space)
            
    # 분리된 단어들을 다시 쉼표(,)로 연결하여 문자열로 반환
    return ", ".join(final_words)

def main():
    # 파일 경로 설정
    input_file = '/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords.csv'
    output_file = '/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv'
    
    if not os.path.exists(input_file):
        print(f"파일을 찾을 수 없습니다: {input_file}")
        return

    # CSV 파일 읽기
    try:
        df = pd.read_csv(input_file)
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    if 'keywords' not in df.columns:
        print("CSV 파일에 'keywords' 컬럼이 없습니다.")
        return

    # === 핵심 변경 부분: 띄어쓰기 제거 함수 적용 ===
    df['keywords_v2'] = df['keywords'].apply(remove_spaces)

    # 결과 확인 출력
    print("데이터 변환 완료! 변환된 샘플 5개를 확인합니다:")
    print(df[['keywords', 'keywords_v2']].head())

    # 새로운 CSV 파일로 저장
    try:
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n새로운 파일이 성공적으로 저장되었습니다:\n{output_file}")
    except Exception as e:
        print(f"파일을 저장하는 중 오류가 발생했습니다: {e}")

if __name__ == "__main__":
    main()


데이터 변환 완료! 변환된 샘플 5개를 확인합니다:
                                            keywords  \
0  럭셔리, 스테이, 리조트, 아만 다리, 호시노야 가루이자와, 스테이폴리오, 인플루언...   
1    롯데리아, 나폴리, 감자연구소, 디저트, 쥐포튀김, 떼리앙, 캐릭터, 띠부씰, 콜라보   
2          국밥, 뉴웨이브, MZ세대, 힙함, 감성, 이색 식재료, 플레이팅, 한 끼   
3      성수, 합정, 홍대, 을지로, 데이트, 카페, 핫플레이스, LP, 퇴근후, 아지트   
4         오이, 소금빵, 디저트, 샌드위치, 바게트, 고리, 슬로우타운, 베이커리호프   

                                         keywords_v2  
0  럭셔리, 스테이, 리조트, 아만다리, 호시노야가루이자와, 스테이폴리오, 인플루언서,...  
1    롯데리아, 나폴리, 감자연구소, 디저트, 쥐포튀김, 떼리앙, 캐릭터, 띠부씰, 콜라보  
2            국밥, 뉴웨이브, MZ세대, 힙함, 감성, 이색식재료, 플레이팅, 한끼  
3      성수, 합정, 홍대, 을지로, 데이트, 카페, 핫플레이스, LP, 퇴근후, 아지트  
4         오이, 소금빵, 디저트, 샌드위치, 바게트, 고리, 슬로우타운, 베이커리호프  

새로운 파일이 성공적으로 저장되었습니다:
/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv


In [9]:
# 1. 데이터가 잘 들어왔는지 먼저 확인

file_path = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
column_name = "keywords_v2"


print(f"데이터 로드 중: {file_path}")
df = pd.read_csv(file_path)

print(f"데이터 행 개수: {len(df)}")
print(f"첫 번째 행 데이터 타입: {type(df['keywords'].iloc[0])}")
print("첫 번째 행 실제값:", repr(df['keywords_v2'].iloc[0]))


데이터 로드 중: /Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv
데이터 행 개수: 1037
첫 번째 행 데이터 타입: <class 'str'>
첫 번째 행 실제값: '럭셔리, 스테이, 리조트, 아만다리, 호시노야가루이자와, 스테이폴리오, 인플루언서, 여행, 숙박'


In [1]:
import pandas as pd
import networkx as nx
from itertools import combinations
import ast
import os


def safe_eval(x):
    """
    다양한 형태의 키워드 데이터를 파이썬 리스트 객체로 안전하게 변환합니다.
    - 형태 A: "['디저트', '당충전']"  → ast.literal_eval 처리
    - 형태 B: "디저트, 당충전, 재충전" → split(',') 처리  ← 현재 데이터 형식
    - 형태 C: "[디저트, 당충전]"       → 대괄호 제거 후 split(',') 처리
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        # 형태 A 시도
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            # 형태 C: 대괄호 제거 후 형태 B와 동일하게 처리
            cleaned = x.strip('[]')
            # 형태 B: 쉼표 기준으로 분리하고 공백 제거
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


def main():
    # 1. 데이터 로드 및 전처리
    file_path = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    column_name = "keywords_v2"

    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return

    print(f"데이터 로드 중: {file_path}")
    df = pd.read_csv(file_path)
    print(f"데이터 행 개수: {len(df)}")

    if column_name not in df.columns:
        print(f"오류: '{column_name}' 컬럼이 데이터프레임에 존재하지 않습니다.")
        return

    # 데이터 타입 변환 (문자열 -> 리스트)
    df[column_name] = df[column_name].apply(safe_eval)

    # 변환 결과 샘플 확인
    print("\n[변환 결과 샘플]")
    for i, val in enumerate(df[column_name].head(3)):
        print(f"  [{i}] {val}")

    # 2. 고빈도 불용어 처리 (전체 게시물의 30% 이상 등장 키워드 제거)
    from collections import Counter

    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):  # 게시물당 1회만 카운트
            kw_doc_freq[kw] += 1

    threshold = total_docs * 0.30
    stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= threshold}

    if stopwords:
        print(f"\n[고빈도 불용어 제거] {len(stopwords)}개: {stopwords}")
    else:
        print("\n[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)")

    # 3. 네트워크 모델링 (NetworkX)
    print("\n동시 출현 네트워크(Co-occurrence Network) 생성 중...")
    G = nx.Graph()

    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue

        # 불용어 제거 후 유효 키워드만 추출
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue

        # 한 게시물 내 단어들로 가능한 모든 2개 조합 추출
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    # 4. 결과 검증 및 통계 출력
    print("-" * 50)
    print(f"네트워크 구축 완료!")
    print(f"전체 노드(키워드) 수: {G.number_of_nodes()}")
    print(f"전체 엣지(연결) 수:   {G.number_of_edges()}")
    print("-" * 50)

    if G.number_of_nodes() == 0:
        print("네트워크가 비어 있습니다. 데이터 형식을 다시 확인하세요.")
        return

    # 가중치(동시 출현 빈도) 기준 상위 10개 단어 쌍 출력
    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    print("\n상위 10개 단어 쌍 (동시 출현 빈도 기준):")
    for u, v, data in sorted_edges[:10]:
        print(f"  [{u}] - [{v}] : {data['weight']}회")

    # 네트워크 기본 통계
    print("\n[네트워크 기본 통계]")
    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    print(f"  평균 연결 중심성 (Avg Degree): {avg_degree:.2f}")
    print(f"  네트워크 밀도 (Density):       {nx.density(G):.4f}")
    # 밀도 해석 안내
    density = nx.density(G)
    if density < 0.01:
        print("  → 희소(Sparse) 네트워크 — 정상 범위")
    elif density < 0.1:
        print("  → 보통 밀도 — 정상 범위")
    else:
        print("  → 고밀도 — 불용어 추가 처리 검토 권장")


if __name__ == "__main__":
    main()

데이터 로드 중: /Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv
데이터 행 개수: 1037

[변환 결과 샘플]
  [0] ['럭셔리', '스테이', '리조트', '아만다리', '호시노야가루이자와', '스테이폴리오', '인플루언서', '여행', '숙박']
  [1] ['롯데리아', '나폴리', '감자연구소', '디저트', '쥐포튀김', '떼리앙', '캐릭터', '띠부씰', '콜라보']
  [2] ['국밥', '뉴웨이브', 'MZ세대', '힙함', '감성', '이색식재료', '플레이팅', '한끼']

[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)

동시 출현 네트워크(Co-occurrence Network) 생성 중...
--------------------------------------------------
네트워크 구축 완료!
전체 노드(키워드) 수: 3982
전체 엣지(연결) 수:   30508
--------------------------------------------------

상위 10개 단어 쌍 (동시 출현 빈도 기준):
  [카페] - [맛집] : 70회
  [디저트] - [카페] : 59회
  [카페] - [브런치] : 43회
  [여행] - [카페] : 41회
  [카페] - [커피] : 40회
  [감성] - [카페] : 31회
  [카페] - [F&B] : 28회
  [카페] - [빈티지] : 28회
  [카페] - [힐링] : 27회
  [디저트] - [커피] : 26회

[네트워크 기본 통계]
  평균 연결 중심성 (Avg Degree): 15.32
  네트워크 밀도 (Density):       0.0038
  → 희소(Sparse) 네트워크 — 정상 범위


In [3]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os


# ============================================================
# 유틸리티 함수
# ============================================================

def safe_eval(x):
    """
    다양한 형태의 키워드 데이터를 파이썬 리스트 객체로 안전하게 변환합니다.
    - 형태 A: "['디저트', '당충전']"  → ast.literal_eval 처리
    - 형태 B: "디저트, 당충전, 재충전" → split(',') 처리  ← 현재 데이터 형식
    - 형태 C: "[디저트, 당충전]"       → 대괄호 제거 후 split(',') 처리
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip('[]')
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


# ============================================================
# Step 1. 데이터 로드 및 동시 출현 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("=" * 55)
    print("[Step 1] 데이터 로드 및 네트워크 생성")
    print("=" * 55)

    # 1-1. 데이터 로드
    if not os.path.exists(file_path):
        print(f"오류: 파일을 찾을 수 없습니다 → {file_path}")
        return None

    df = pd.read_csv(file_path)
    print(f"데이터 로드 완료 | 행 개수: {len(df)}")

    if column_name not in df.columns:
        print(f"오류: '{column_name}' 컬럼이 존재하지 않습니다.")
        return None

    # 1-2. 키워드 컬럼 리스트 변환
    df[column_name] = df[column_name].apply(safe_eval)

    print("\n[변환 결과 샘플]")
    for i, val in enumerate(df[column_name].head(3)):
        print(f"  [{i}] {val}")

    # 1-3. 고빈도 불용어 처리 (전체 게시물의 30% 이상 등장 키워드 제거)
    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    threshold = total_docs * 0.30
    stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= threshold}

    if stopwords:
        print(f"\n[고빈도 불용어 제거] {len(stopwords)}개: {stopwords}")
    else:
        print("\n[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)")

    # 1-4. 동시 출현 네트워크 생성
    print("\n동시 출현 네트워크(Co-occurrence Network) 생성 중...")
    G = nx.Graph()

    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue

        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue

        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    # 1-5. 네트워크 기본 통계 출력
    print("-" * 55)
    print(f"네트워크 구축 완료!")
    print(f"  전체 노드(키워드) 수 : {G.number_of_nodes()}")
    print(f"  전체 엣지(연결) 수   : {G.number_of_edges()}")

    if G.number_of_nodes() == 0:
        print("네트워크가 비어 있습니다. 데이터 형식을 확인하세요.")
        return None

    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    density = nx.density(G)
    print(f"  평균 연결 중심성     : {avg_degree:.2f}")
    print(f"  네트워크 밀도        : {density:.4f}", end=" ")
    if density < 0.01:
        print("→ 희소(Sparse) 네트워크 — 정상 범위")
    elif density < 0.1:
        print("→ 보통 밀도 — 정상 범위")
    else:
        print("→ 고밀도 — 불용어 추가 처리 검토 권장")

    print("\n[상위 10개 단어 쌍 (동시 출현 빈도 기준)]")
    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    for u, v, data in sorted_edges[:10]:
        print(f"  [{u}] - [{v}] : {data['weight']}회")

    return G


# ============================================================
# Step 2. 중심성 지표 계산
# ============================================================

def analyze_centrality(G, save_path="centrality_results_2025.csv"):
    print("\n" + "=" * 55)
    print("[Step 2] 중심성 지표 계산")
    print("=" * 55)

    # 2-1. 연결 중심성: 대중적 대세 트렌드
    print(" - 연결 중심성 계산 중...")
    deg_cent = nx.degree_centrality(G)

    # 2-2. 고유벡터 중심성: 파워 트렌드
    # weight='weight' 적용 — 동시 출현 빈도(친밀도)를 반영하여 계산
    print(" - 고유벡터 중심성 계산 중...")
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        print("   (⚠️ 수렴하지 않아 가중치 없이 계산합니다.)")
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    # 2-3. 매개 중심성: 도메인 간 융합 허브
    # weight 미사용 이유: NetworkX는 weight를 '거리(비용)'로 해석하므로
    # 동시 출현 빈도(친밀도) 기반 네트워크에서는 구조적 위치만으로 계산하는 것이 정확함
    print(" - 매개 중심성 계산 중... (⏳ 1~3분 소요될 수 있습니다)")
    bet_cent = nx.betweenness_centrality(G)

    # 2-4. 결과 데이터프레임 통합
    print(" - 결과를 데이터프레임으로 정리 중...")
    centrality_df = pd.DataFrame({
        'Keyword'           : list(G.nodes()),
        'Degree (대세)'     : [deg_cent[node] for node in G.nodes()],
        'Eigenvector (파워)': [eig_cent[node] for node in G.nodes()],
        'Betweenness (융합)': [bet_cent[node] for node in G.nodes()]
    })
    centrality_df = centrality_df.sort_values(
        by='Degree (대세)', ascending=False
    ).reset_index(drop=True)

    # 2-5. 결과 저장
    centrality_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"\n중심성 분석 완료! 결과 저장 → '{save_path}'")

    # 2-6. 지표별 Top 5 출력
    print("\n[각 지표별 Top 5 키워드]")
    top_degree = centrality_df.sort_values(
        by='Degree (대세)', ascending=False)['Keyword'].head(5).tolist()
    top_eigen = centrality_df.sort_values(
        by='Eigenvector (파워)', ascending=False)['Keyword'].head(5).tolist()
    top_between = centrality_df.sort_values(
        by='Betweenness (융합)', ascending=False)['Keyword'].head(5).tolist()

    print(f"  대세  (Degree)      Top 5: {top_degree}")
    print(f"  파워  (Eigenvector) Top 5: {top_eigen}")
    print(f"  융합허브(Betweenness) Top 5: {top_between}")

    return centrality_df


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":
    FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    COLUMN_NAME = "keywords_v2"
    SAVE_PATH   = "centrality_results_2025.csv"

    # Step 1
    G = build_network(FILE_PATH, COLUMN_NAME)

    # Step 2
    if G is not None:
        centrality_df = analyze_centrality(G, SAVE_PATH)

[Step 1] 데이터 로드 및 네트워크 생성
데이터 로드 완료 | 행 개수: 1037

[변환 결과 샘플]
  [0] ['럭셔리', '스테이', '리조트', '아만다리', '호시노야가루이자와', '스테이폴리오', '인플루언서', '여행', '숙박']
  [1] ['롯데리아', '나폴리', '감자연구소', '디저트', '쥐포튀김', '떼리앙', '캐릭터', '띠부씰', '콜라보']
  [2] ['국밥', '뉴웨이브', 'MZ세대', '힙함', '감성', '이색식재료', '플레이팅', '한끼']

[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)

동시 출현 네트워크(Co-occurrence Network) 생성 중...
-------------------------------------------------------
네트워크 구축 완료!
  전체 노드(키워드) 수 : 3982
  전체 엣지(연결) 수   : 30508
  평균 연결 중심성     : 15.32
  네트워크 밀도        : 0.0038 → 희소(Sparse) 네트워크 — 정상 범위

[상위 10개 단어 쌍 (동시 출현 빈도 기준)]
  [카페] - [맛집] : 70회
  [디저트] - [카페] : 59회
  [카페] - [브런치] : 43회
  [여행] - [카페] : 41회
  [카페] - [커피] : 40회
  [감성] - [카페] : 31회
  [카페] - [F&B] : 28회
  [카페] - [빈티지] : 28회
  [카페] - [힐링] : 27회
  [디저트] - [커피] : 26회

[Step 2] 중심성 지표 계산
 - 연결 중심성 계산 중...
 - 고유벡터 중심성 계산 중...
 - 매개 중심성 계산 중... (⏳ 1~3분 소요될 수 있습니다)
 - 결과를 데이터프레임으로 정리 중...

중심성 분석 완료! 결과 저장 → 'centrality_results_2025.csv'

[각 지표별 Top 5 키워드]
  대세  (Degree)      Top 5: ['

In [4]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os


# ============================================================
# 유틸리티 함수
# ============================================================

def safe_eval(x):
    """
    다양한 형태의 키워드 데이터를 파이썬 리스트 객체로 안전하게 변환합니다.
    - 형태 A: "['디저트', '당충전']"  → ast.literal_eval 처리
    - 형태 B: "디저트, 당충전, 재충전" → split(',') 처리  ← 현재 데이터 형식
    - 형태 C: "[디저트, 당충전]"       → 대괄호 제거 후 split(',') 처리
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip('[]')
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


# ============================================================
# Step 1. 데이터 로드 및 동시 출현 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("=" * 55)
    print("[Step 1] 데이터 로드 및 네트워크 생성")
    print("=" * 55)

    # 1-1. 데이터 로드
    if not os.path.exists(file_path):
        print(f"오류: 파일을 찾을 수 없습니다 → {file_path}")
        return None

    df = pd.read_csv(file_path)
    print(f"데이터 로드 완료 | 행 개수: {len(df)}")

    if column_name not in df.columns:
        print(f"오류: '{column_name}' 컬럼이 존재하지 않습니다.")
        return None

    # 1-2. 키워드 컬럼 리스트 변환
    df[column_name] = df[column_name].apply(safe_eval)

    print("\n[변환 결과 샘플]")
    for i, val in enumerate(df[column_name].head(3)):
        print(f"  [{i}] {val}")

    # 1-3. 고빈도 불용어 처리 (전체 게시물의 30% 이상 등장 키워드 제거)
    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    threshold = total_docs * 0.30
    high_freq_stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= threshold}

    # 매거진명, 플랫폼명 등 분석 노이즈가 되는 고유명사를 수동으로 추가
    manual_stopwords = {
        '뉴뉴',   # 매거진 플랫폼명
    }

    stopwords = high_freq_stopwords | manual_stopwords

    print(f"\n[고빈도 불용어] {len(high_freq_stopwords)}개 (임계값: 전체 게시물의 30%)")
    print(f"[수동 불용어]   {len(manual_stopwords)}개: {manual_stopwords}")
    print(f"[불용어 합계]   총 {len(stopwords)}개 제거")

    # 1-4. 동시 출현 네트워크 생성
    print("\n동시 출현 네트워크(Co-occurrence Network) 생성 중...")
    G = nx.Graph()

    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue

        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue

        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    # 1-5. 네트워크 기본 통계 출력
    print("-" * 55)
    print(f"네트워크 구축 완료!")
    print(f"  전체 노드(키워드) 수 : {G.number_of_nodes()}")
    print(f"  전체 엣지(연결) 수   : {G.number_of_edges()}")

    if G.number_of_nodes() == 0:
        print("네트워크가 비어 있습니다. 데이터 형식을 확인하세요.")
        return None

    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    density = nx.density(G)
    print(f"  평균 연결 중심성     : {avg_degree:.2f}")
    print(f"  네트워크 밀도        : {density:.4f}", end=" ")
    if density < 0.01:
        print("→ 희소(Sparse) 네트워크 — 정상 범위")
    elif density < 0.1:
        print("→ 보통 밀도 — 정상 범위")
    else:
        print("→ 고밀도 — 불용어 추가 처리 검토 권장")

    print("\n[상위 10개 단어 쌍 (동시 출현 빈도 기준)]")
    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    for u, v, data in sorted_edges[:10]:
        print(f"  [{u}] - [{v}] : {data['weight']}회")

    return G


# ============================================================
# Step 2. 중심성 지표 계산
# ============================================================

def analyze_centrality(G, save_path="centrality_results_2025.csv"):
    print("\n" + "=" * 55)
    print("[Step 2] 중심성 지표 계산")
    print("=" * 55)

    # 2-1. 연결 중심성: 대중적 대세 트렌드
    print(" - 연결 중심성 계산 중...")
    deg_cent = nx.degree_centrality(G)

    # 2-2. 고유벡터 중심성: 파워 트렌드
    # weight='weight' 적용 — 동시 출현 빈도(친밀도)를 반영하여 계산
    print(" - 고유벡터 중심성 계산 중...")
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        print("   (⚠️ 수렴하지 않아 가중치 없이 계산합니다.)")
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    # 2-3. 매개 중심성: 도메인 간 융합 허브
    # weight 미사용 이유: NetworkX는 weight를 '거리(비용)'로 해석하므로
    # 동시 출현 빈도(친밀도) 기반 네트워크에서는 구조적 위치만으로 계산하는 것이 정확함
    print(" - 매개 중심성 계산 중... (⏳ 1~3분 소요될 수 있습니다)")
    bet_cent = nx.betweenness_centrality(G)

    # 2-4. 결과 데이터프레임 통합
    print(" - 결과를 데이터프레임으로 정리 중...")
    centrality_df = pd.DataFrame({
        'Keyword'           : list(G.nodes()),
        'Degree (대세)'     : [deg_cent[node] for node in G.nodes()],
        'Eigenvector (파워)': [eig_cent[node] for node in G.nodes()],
        'Betweenness (융합)': [bet_cent[node] for node in G.nodes()]
    })
    centrality_df = centrality_df.sort_values(
        by='Degree (대세)', ascending=False
    ).reset_index(drop=True)

    # 2-5. 결과 저장
    centrality_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"\n중심성 분석 완료! 결과 저장 → '{save_path}'")

    # 2-6. 지표별 Top 5 출력
    print("\n[각 지표별 Top 5 키워드]")
    top_degree = centrality_df.sort_values(
        by='Degree (대세)', ascending=False)['Keyword'].head(5).tolist()
    top_eigen = centrality_df.sort_values(
        by='Eigenvector (파워)', ascending=False)['Keyword'].head(5).tolist()
    top_between = centrality_df.sort_values(
        by='Betweenness (융합)', ascending=False)['Keyword'].head(5).tolist()

    print(f"  대세  (Degree)      Top 5: {top_degree}")
    print(f"  파워  (Eigenvector) Top 5: {top_eigen}")
    print(f"  융합허브(Betweenness) Top 5: {top_between}")

    return centrality_df


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":
    FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    COLUMN_NAME = "keywords_v2"
    SAVE_PATH   = "centrality_results_2025.csv"

    # Step 1
    G = build_network(FILE_PATH, COLUMN_NAME)

    # Step 2
    if G is not None:
        centrality_df = analyze_centrality(G, SAVE_PATH)

[Step 1] 데이터 로드 및 네트워크 생성
데이터 로드 완료 | 행 개수: 1037

[변환 결과 샘플]
  [0] ['럭셔리', '스테이', '리조트', '아만다리', '호시노야가루이자와', '스테이폴리오', '인플루언서', '여행', '숙박']
  [1] ['롯데리아', '나폴리', '감자연구소', '디저트', '쥐포튀김', '떼리앙', '캐릭터', '띠부씰', '콜라보']
  [2] ['국밥', '뉴웨이브', 'MZ세대', '힙함', '감성', '이색식재료', '플레이팅', '한끼']

[고빈도 불용어] 0개 (임계값: 전체 게시물의 30%)
[수동 불용어]   1개: {'뉴뉴'}
[불용어 합계]   총 1개 제거

동시 출현 네트워크(Co-occurrence Network) 생성 중...
-------------------------------------------------------
네트워크 구축 완료!
  전체 노드(키워드) 수 : 3981
  전체 엣지(연결) 수   : 30194
  평균 연결 중심성     : 15.17
  네트워크 밀도        : 0.0038 → 희소(Sparse) 네트워크 — 정상 범위

[상위 10개 단어 쌍 (동시 출현 빈도 기준)]
  [카페] - [맛집] : 70회
  [디저트] - [카페] : 59회
  [카페] - [브런치] : 43회
  [여행] - [카페] : 41회
  [카페] - [커피] : 40회
  [감성] - [카페] : 31회
  [카페] - [F&B] : 28회
  [카페] - [빈티지] : 28회
  [카페] - [힐링] : 27회
  [디저트] - [커피] : 26회

[Step 2] 중심성 지표 계산
 - 연결 중심성 계산 중...
 - 고유벡터 중심성 계산 중...
 - 매개 중심성 계산 중... (⏳ 1~3분 소요될 수 있습니다)
 - 결과를 데이터프레임으로 정리 중...

중심성 분석 완료! 결과 저장 → 'centrality_results_2025.csv'

[각 지표별 To

In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os

# Louvain 알고리즘 (python-louvain 패키지)
try:
    import community as community_louvain
except ImportError:
    print("python-louvain 패키지가 없습니다. 설치 중...")
    os.system("pip install python-louvain")
    import community as community_louvain

# 인터랙티브 시각화 (pyvis 패키지)
try:
    from pyvis.network import Network
except ImportError:
    print("pyvis 패키지가 없습니다. 설치 중...")
    os.system("pip install pyvis")
    from pyvis.network import Network


# ============================================================
# 유틸리티 함수
# ============================================================

def safe_eval(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip('[]')
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


# ============================================================
# Step 1. 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("[Step 1] 네트워크 생성 중...")
    df = pd.read_csv(file_path)
    df[column_name] = df[column_name].apply(safe_eval)

    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    high_freq_stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= total_docs * 0.30}
    manual_stopwords = {
        '뉴뉴',  # 매거진 플랫폼명
    }
    stopwords = high_freq_stopwords | manual_stopwords
    print(f"  불용어 제거: 총 {len(stopwords)}개")

    G = nx.Graph()
    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    print(f"  노드: {G.number_of_nodes()}개 | 엣지: {G.number_of_edges()}개")
    return G


# ============================================================
# Step 4. Louvain 메가트렌드 군집화
# ============================================================

def detect_communities(G):
    print("\n[Step 4] Louvain 커뮤니티 탐지 중...")
    partition = community_louvain.best_partition(G, weight='weight', random_state=42)

    community_sizes = Counter(partition.values())
    n_communities = len(community_sizes)
    print(f"  발견된 군집 수: {n_communities}개")

    deg_cent = nx.degree_centrality(G)
    print("\n[군집별 Top 5 키워드 (연결 중심성 기준)]")
    print("-" * 55)
    for comm_id in sorted(community_sizes, key=community_sizes.get, reverse=True):
        members = [node for node, c in partition.items() if c == comm_id]
        top5 = sorted(members, key=lambda n: deg_cent[n], reverse=True)[:5]
        print(f"  군집 {comm_id:2d} ({len(members):4d}개): {top5}")
    print("-" * 55)

    return partition


# ============================================================
# 시각화: 전체 네트워크 인터랙티브 HTML
# ============================================================

# 군집 ID → 색상 팔레트 (20가지)
PALETTE = [
    "#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6",
    "#1ABC9C", "#E67E22", "#34495E", "#E91E63", "#00BCD4",
    "#8BC34A", "#FF5722", "#607D8B", "#795548", "#FF9800",
    "#673AB7", "#009688", "#F44336", "#2196F3", "#4CAF50",
]


def build_visualization(G, partition, save_path="trend_network_full.html"):
    print("\n[시각화] 전체 네트워크 HTML 생성 중...")

    # 연결 중심성 (대세) → 노드 크기
    deg_cent = nx.degree_centrality(G)

    # 고유벡터 중심성 (파워) → 노드 테두리 두께
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    # 고유벡터 중심성 정규화 → 테두리 두께 1~10px
    eig_values = list(eig_cent.values())
    eig_min, eig_max = min(eig_values), max(eig_values)

    def eig_to_border(val):
        normalized = (val - eig_min) / (eig_max - eig_min + 1e-9)
        return 1 + normalized * 9  # 1px ~ 10px

    community_sizes = Counter(partition.values())

    # 군집 ID를 크기 순으로 정렬하여 색상 인덱스 부여
    comm_rank = {
        comm_id: rank
        for rank, (comm_id, _) in enumerate(
            sorted(community_sizes.items(), key=lambda x: x[1], reverse=True)
        )
    }

    # pyvis 네트워크 설정
    net = Network(
        height="950px",
        width="100%",
        bgcolor="#1a1a2e",
        font_color="#ffffff",
        notebook=False
    )

    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -50,
          "centralGravity": 0.005,
          "springLength": 100,
          "springConstant": 0.08,
          "damping": 0.4,
          "avoidOverlap": 0.5
        },
        "maxVelocity": 50,
        "minVelocity": 0.1,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {
          "enabled": true,
          "iterations": 200
        }
      },
      "nodes": {
        "shape": "dot",
        "font": {
          "size": 12,
          "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"
        },
        "borderWidthSelected": 4
      },
      "edges": {
        "smooth": {
          "enabled": true,
          "type": "continuous"
        }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "hideEdgesOnDrag": true,
        "navigationButtons": true,
        "keyboard": true
      }
    }
    """)

    # 노드 추가
    for node in G.nodes():
        comm_id      = partition[node]
        color_idx    = comm_rank.get(comm_id, 0) % len(PALETTE)
        color        = PALETTE[color_idx]
        degree       = deg_cent[node]
        border_width = eig_to_border(eig_cent[node])

        # 연결 중심성 → 노드 크기 (5~40px)
        size = min(5 + degree * 120, 40)

        net.add_node(
            node,
            label=node,
            color={
                "background": color,
                "border":     "#ffffff",
                "highlight":  {"background": color, "border": "#FFD700"},
                "hover":      {"background": color, "border": "#FFD700"},
            },
            size=size,
            borderWidth=border_width,
            title=(
                f"키워드: {node}\n"
                f"군집: {comm_id}\n"
                f"─────────────────\n"
                f"크기   → 대세  (연결 중심성):     {deg_cent[node]:.4f}\n"
                f"테두리 → 파워  (고유벡터 중심성): {eig_cent[node]:.4f}\n"
                f"─────────────────\n"
                f"군집 크기: {community_sizes[comm_id]}개"
            )
        )

    # 엣지 추가 (가중치 낮은 엣지는 흐리게)
    max_weight = max(d['weight'] for _, _, d in G.edges(data=True))
    for u, v, data in G.edges(data=True):
        w       = data['weight']
        opacity = 0.05 + 0.4 * (w / max_weight)
        width   = 0.5 + 2.5 * (w / max_weight)
        net.add_edge(
            u, v,
            value=w,
            width=width,
            color={"opacity": opacity},
            title=f"{u} ↔ {v}\n동시 출현: {w}회"
        )

    net.save_graph(save_path)
    print(f"  저장 완료 → '{save_path}'")
    print("  브라우저에서 파일을 열어 확인하세요.")
    print("  (노드 드래그 · 확대/축소 · 호버 시 지표 확인 가능)")


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":
    FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    COLUMN_NAME = "keywords_v2"
    SAVE_PATH   = "/Users/hajiyoon/workspace/trend_network_full.html"3

    G         = build_network(FILE_PATH, COLUMN_NAME)
    partition = detect_communities(G)
    build_visualization(G, partition, SAVE_PATH)

[Step 1] 네트워크 생성 중...
  불용어 제거: 총 1개
  노드: 3981개 | 엣지: 30194개

[Step 4] Louvain 커뮤니티 탐지 중...
  발견된 군집 수: 34개

[군집별 Top 5 키워드 (연결 중심성 기준)]
-------------------------------------------------------
  군집  1 ( 832개): ['카페', '디저트', '맛집', '커피', '브런치']
  군집  6 ( 451개): ['말차', 'CU', '딸기', '제주', '스타벅스']
  군집  7 ( 394개): ['F&B', '성수', '와인', '연말', '크리스마스']
  군집  8 ( 250개): ['케이크', '겨울', '음식', '가을', '선물']
  군집 10 ( 227개): ['여름', '제철', '달콤', '빙수', '상큼']
  군집  2 ( 168개): ['팝업', '콜라보', '타코', '플리마켓', '홍콩']
  군집  4 ( 161개): ['레트로', '저녁', '점심', '버거', '만두']
  군집 22 ( 149개): ['안주', '한식', '퓨전', '맥주', '연희동']
  군집 12 ( 146개): ['떡볶이', '오뎅', '하이볼', '가성비', '야식']
  군집 18 ( 132개): ['쇼핑', '도쿄', '라멘', '휴식', '족발']
  군집 17 ( 128개): ['일본', '바삭', '일식', '직장인', '키링']
  군집  3 ( 118개): ['빵', '베이글', '빵지순례', '빵순이', '초여름']
  군집 11 ( 115개): ['간식', '추억', '공간', '킷사텐', '제니']
  군집 20 ( 102개): ['건강', '비건', '김밥', '아침', '다이어트']
  군집 19 (  82개): ['흑백요리사', '시즌', '밥친구', '셰프', '핫플']
  군집  5 (  77개): ['당충전', '한남동', '미식', '체험', '독일']
  군집  0

In [8]:
# 기존 코드 실행 후 G, partition이 있는 상태에서 아래 추가 실행

import pandas as pd
import networkx as nx
from collections import Counter

deg_cent = nx.degree_centrality(G)
community_sizes = Counter(partition.values())

rows = []
for comm_id in sorted(community_sizes, key=community_sizes.get, reverse=True):
    members = [node for node, c in partition.items() if c == comm_id]
    top10 = sorted(members, key=lambda n: deg_cent[n], reverse=True)[:10]
    rows.append({
        '군집ID':    comm_id,
        '키워드수':  community_sizes[comm_id],
        'Top10키워드': ', '.join(top10),
        '메가트렌드명': ''  # 나중에 직접 채울 칸
    })

result_df = pd.DataFrame(rows)
result_df.to_csv('/Users/hajiyoon/workspace/megatrend_naming.csv',
                 index=False, encoding='utf-8-sig')
print(result_df[['군집ID','키워드수','Top10키워드']].to_string())

    군집ID  키워드수                                                     Top10키워드
0      1   832                   카페, 디저트, 맛집, 커피, 브런치, 데이트, 여행, 빈티지, 감성, 힐링
1      6   451               말차, CU, 딸기, 제주, 스타벅스, 아이스크림, 피스타치오, 굿즈, 망고, 미국
2      7   394              F&B, 성수, 와인, 연말, 크리스마스, 샌드위치, 파스타, 이자카야, 신상, 피자
3      8   250                   케이크, 겨울, 음식, 가을, 선물, 브랜드, 취향, 친구, 무화과, 디자인
4     10   227                     여름, 제철, 달콤, 빙수, 상큼, 과일, 파르페, 음료, 성심당, 셀럽
5      2   168               팝업, 콜라보, 타코, 플리마켓, 홍콩, 캐릭터, 성수동, 막걸리, 큐레이션, 파리
6      4   161                  레트로, 저녁, 점심, 버거, 만두, 아메리칸, 국밥, 닭갈비, 다방, 사랑방
7     22   149                     안주, 한식, 퓨전, 맥주, 연희동, 음악, 혼술, 햄버거, 맛, 위스키
8     12   146                  떡볶이, 오뎅, 하이볼, 가성비, 야식, 대구, 야장, 시원함, 플라워, 노포
9     18   132                      쇼핑, 도쿄, 라멘, 휴식, 족발, 메뉴, 편집샵, 루틴, 전시, 스시
10    17   128                  일본, 바삭, 일식, 직장인, 키링, 사시미, 감칠맛, 오리지널, 식감, 카츠
11     3   118              빵, 베이글, 빵지순례, 빵순이, 초여름, 슈톨렌, 카라멜, 빵덕후, 호두과자, 천안
12    11   1

In [9]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os

try:
    import community as community_louvain
except ImportError:
    os.system("pip install python-louvain")
    import community as community_louvain

try:
    from pyvis.network import Network
except ImportError:
    os.system("pip install pyvis")
    from pyvis.network import Network


# ============================================================
# 메가트렌드 명명 딕셔너리 (군집ID → 메가트렌드명)
# ============================================================

MEGATREND_NAMES = {
    1:  "감성 카페 라이프",
    6:  "시즌 한정 디저트",
    7:  "핫플레이스 다이닝",
    8:  "선물형 디저트 브랜딩",
    10: "제철 과일 디저트",
    2:  "팝업·콜라보 경험",
    4:  "레트로 로컬 푸드",
    22: "혼술·홈술 문화",
    12: "가성비 야식·노포",
    18: "라이프스타일 쇼핑",
    17: "일식 열풍",
    3:  "빵지순례 문화",
    11: "추억·향수 간식",
    20: "건강식·비건",
    19: "미디어 푸드 콘텐츠",
    5:  "미식 체험",
    0:  "먹방·여행 콘텐츠",
    9:  "골목 술자리 문화",
    23: "수험생·이벤트 특수",
    13: "비주얼 데이트 플레이스",
    14: "프리미엄 파인다이닝",
    15: "도심 SNS 스팟",
    30: "연말연시 이벤트",
    21: "프리미엄 버거 재료",
    28: "시즌 한정 음료",
    27: "럭셔리 브랜드 F&B",
    24: "패션·다이닝 크로스오버",
    26: "로컬 여행 맛집",
    16: "빈티지 라이프스타일",
    31: "인플루언서 맛집",
    25: "간편결제 라이프",
    29: "대학가 음식 문화",
    32: "엔터테인먼트 스팟",
    33: "배달·푸드테크",
}

# ============================================================
# 유틸리티 함수
# ============================================================

def safe_eval(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip('[]')
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


# ============================================================
# Step 1. 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("[Step 1] 네트워크 생성 중...")
    df = pd.read_csv(file_path)
    df[column_name] = df[column_name].apply(safe_eval)

    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    high_freq_stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= total_docs * 0.30}
    manual_stopwords = {'뉴뉴'}
    stopwords = high_freq_stopwords | manual_stopwords
    print(f"  불용어 제거: 총 {len(stopwords)}개")

    G = nx.Graph()
    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    print(f"  노드: {G.number_of_nodes()}개 | 엣지: {G.number_of_edges()}개")
    return G


# ============================================================
# Step 4. Louvain 메가트렌드 군집화
# ============================================================

def detect_communities(G):
    print("\n[Step 4] Louvain 커뮤니티 탐지 중...")
    partition = community_louvain.best_partition(G, weight='weight', random_state=42)
    community_sizes = Counter(partition.values())
    print(f"  발견된 군집 수: {len(community_sizes)}개")
    return partition


# ============================================================
# 시각화
# ============================================================

PALETTE = [
    "#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6",
    "#1ABC9C", "#E67E22", "#34495E", "#E91E63", "#00BCD4",
    "#8BC34A", "#FF5722", "#607D8B", "#795548", "#FF9800",
    "#673AB7", "#009688", "#F44336", "#2196F3", "#4CAF50",
    "#CDDC39", "#FF5252", "#40C4FF", "#69F0AE", "#FFD740",
    "#E040FB", "#00E5FF", "#76FF03", "#FF6D00", "#651FFF",
    "#F50057", "#00B0FF", "#C6FF00", "#FFAB40", "#D500F9",
]


def build_visualization(G, partition, save_path="trend_network_full.html"):
    print("\n[시각화] HTML 생성 중...")

    deg_cent = nx.degree_centrality(G)

    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    eig_values = list(eig_cent.values())
    eig_min, eig_max = min(eig_values), max(eig_values)

    def eig_to_border(val):
        normalized = (val - eig_min) / (eig_max - eig_min + 1e-9)
        return 1 + normalized * 9

    community_sizes = Counter(partition.values())

    comm_rank = {
        comm_id: rank
        for rank, (comm_id, _) in enumerate(
            sorted(community_sizes.items(), key=lambda x: x[1], reverse=True)
        )
    }

    net = Network(
        height="950px",
        width="100%",
        bgcolor="#1a1a2e",
        font_color="#ffffff",
        notebook=False
    )

    # 안정화 후 물리 엔진 OFF + 클릭 시 연결 노드만 표시
    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -50,
          "centralGravity": 0.005,
          "springLength": 100,
          "springConstant": 0.08,
          "damping": 0.4,
          "avoidOverlap": 0.5
        },
        "maxVelocity": 50,
        "minVelocity": 0.75,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {
          "enabled": true,
          "iterations": 300,
          "updateInterval": 50,
          "fit": true
        }
      },
      "nodes": {
        "shape": "dot",
        "font": {
          "size": 12,
          "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"
        },
        "borderWidthSelected": 4
      },
      "edges": {
        "smooth": {
          "enabled": true,
          "type": "continuous"
        }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "hideEdgesOnDrag": true,
        "navigationButtons": true,
        "keyboard": true,
        "selectConnectedEdges": true
      }
    }
    """)

    for node in G.nodes():
        comm_id      = partition[node]
        megatrend    = MEGATREND_NAMES.get(comm_id, f"군집 {comm_id}")
        color_idx    = comm_rank.get(comm_id, 0) % len(PALETTE)
        color        = PALETTE[color_idx]
        degree       = deg_cent[node]
        border_width = eig_to_border(eig_cent[node])
        size         = min(5 + degree * 120, 40)

        net.add_node(
            node,
            label=node,
            color={
                "background": color,
                "border":     "#ffffff",
                "highlight":  {"background": color, "border": "#FFD700"},
                "hover":      {"background": color, "border": "#FFD700"},
            },
            size=size,
            borderWidth=border_width,
            title=(
                f"키워드: {node}\n"
                f"메가트렌드: {megatrend}\n"
                f"─────────────────\n"
                f"크기   → 대세  (연결 중심성):     {deg_cent[node]:.4f}\n"
                f"테두리 → 파워  (고유벡터 중심성): {eig_cent[node]:.4f}\n"
                f"─────────────────\n"
                f"군집 크기: {community_sizes[comm_id]}개"
            ),
            group=str(comm_id)
        )

    max_weight = max(d['weight'] for _, _, d in G.edges(data=True))
    for u, v, data in G.edges(data=True):
        w       = data['weight']
        opacity = 0.05 + 0.4 * (w / max_weight)
        width   = 0.5 + 2.5 * (w / max_weight)
        net.add_edge(
            u, v,
            value=w,
            width=width,
            color={"opacity": opacity},
            title=f"{u} ↔ {v}\n동시 출현: {w}회"
        )

    # HTML 저장 후 JS 주입
    net.save_graph(save_path)

    with open(save_path, 'r', encoding='utf-8') as f:
        html = f.read()

    # 1) 안정화 완료 후 물리 엔진 OFF (꿈틀거림 제거)
    # 2) 노드 클릭 시 해당 노드 + 연결된 노드/엣지만 표시, 나머지 희미하게
    custom_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {

  // 안정화 완료 후 물리 엔진 OFF
  network.once("stabilized", function() {
    network.setOptions({ physics: { enabled: false } });
  });

  // 노드 클릭 시 연결 노드만 강조
  network.on("click", function(params) {
    if (params.nodes.length > 0) {
      var selectedNode = params.nodes[0];
      var connectedNodes = network.getConnectedNodes(selectedNode);
      var connectedEdges = network.getConnectedEdges(selectedNode);

      var allNodes = nodes.get();
      var allEdges = edges.get();

      // 연결되지 않은 노드는 희미하게
      var updateNodes = allNodes.map(function(n) {
        if (n.id === selectedNode || connectedNodes.indexOf(n.id) !== -1) {
          return { id: n.id, opacity: 1.0 };
        } else {
          return { id: n.id, opacity: 0.05 };
        }
      });

      // 연결되지 않은 엣지는 희미하게
      var updateEdges = allEdges.map(function(e) {
        if (connectedEdges.indexOf(e.id) !== -1) {
          return { id: e.id, color: { opacity: 0.9 } };
        } else {
          return { id: e.id, color: { opacity: 0.02 } };
        }
      });

      nodes.update(updateNodes);
      edges.update(updateEdges);

    } else {
      // 빈 곳 클릭 시 원래대로 복원
      var allNodes = nodes.get();
      var allEdges = edges.get();

      var resetNodes = allNodes.map(function(n) {
        return { id: n.id, opacity: 1.0 };
      });
      var resetEdges = allEdges.map(function(e) {
        return { id: e.id, color: { opacity: null } };
      });

      nodes.update(resetNodes);
      edges.update(resetEdges);
    }
  });
});
</script>
"""

    # </body> 직전에 JS 삽입
    html = html.replace('</body>', custom_js + '\n</body>')

    with open(save_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f"  저장 완료 → '{save_path}'")
    print("  [사용법]")
    print("  - 노드 클릭: 연결된 노드만 강조, 나머지 희미해짐")
    print("  - 빈 곳 클릭: 전체 복원")
    print("  - 호버: 키워드·메가트렌드명·지표 확인")
    print("  - 네트워크는 안정화 후 자동으로 고정됩니다")


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":
    FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    COLUMN_NAME = "keywords_v2"
    SAVE_PATH   = "/Users/hajiyoon/workspace/trend_network_full.html"

    G         = build_network(FILE_PATH, COLUMN_NAME)
    partition = detect_communities(G)
    build_visualization(G, partition, SAVE_PATH)

[Step 1] 네트워크 생성 중...
  불용어 제거: 총 1개
  노드: 3981개 | 엣지: 30194개

[Step 4] Louvain 커뮤니티 탐지 중...
  발견된 군집 수: 34개

[시각화] HTML 생성 중...
  저장 완료 → '/Users/hajiyoon/workspace/trend_network_full.html'
  [사용법]
  - 노드 클릭: 연결된 노드만 강조, 나머지 희미해짐
  - 빈 곳 클릭: 전체 복원
  - 호버: 키워드·메가트렌드명·지표 확인
  - 네트워크는 안정화 후 자동으로 고정됩니다


In [11]:
import pandas as pd

df = pd.read_csv("/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv")

# 전체 컬럼 확인
print(df.columns.tolist())

print(df['date'].head(10))
print("\n타입:", df['date'].dtype)

['post_id', 'date', 'timestamp', 'title', 'body', 'likes', 'comments', 'url', '광고 미포함', 'keywords', 'keywords_v2']
0    2025-04-21
1    2025-04-21
2    2025-04-21
3    2025-04-22
4    2025-04-22
5    2025-04-22
6    2025-04-23
7    2025-04-23
8    2025-04-24
9    2025-04-24
Name: date, dtype: str

타입: str


In [12]:
import pandas as pd

df = pd.read_csv("/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv")

df['date'] = pd.to_datetime(df['date'])

print("시작일:", df['date'].min())
print("종료일:", df['date'].max())
print("총 기간:", (df['date'].max() - df['date'].min()).days, "일")
print("\n월별 게시물 수:")
print(df.groupby(df['date'].dt.to_period('M')).size().to_string())

시작일: 2025-04-21 00:00:00
종료일: 2025-12-31 00:00:00
총 기간: 254 일

월별 게시물 수:
date
2025-04     34
2025-05    104
2025-06    106
2025-07    130
2025-08    125
2025-09    115
2025-10    122
2025-11    142
2025-12    159
Freq: M


In [2]:
import pandas as pd
import os
df_merged = pd.read_csv("/Users/hajiyoon/workspace/data/processed/knewnew_full_2025_v2.csv")
print(df_merged.head(3))

       post_id        date                 timestamp  \
0  3.61551e+18  2025-04-21  2025-04-21T08:00:00.000Z   
1  3.61533e+18  2025-04-21  2025-04-21T02:00:00.000Z   
2   3.6153e+18  2025-04-21  2025-04-21T01:01:39.000Z   

                                     title  \
0  #AD #EVENT 안목 있는 50인이 고른 베스트 스테이 모음.zip   
1                 #제작지원 🍟 롯데리아에 작은 거 온다(?)   
2  😎 이게 요즘 뉴웨이브 국밥? MZ세대의 힙한 감성이 녹아든 국밥집들.   

                                                body  likes  comments  \
0  🏡 스테이폴리오 10주년 기념!\n배우 한효주, 아티스트 마이큐, 우아한형제들 창업...    366       2.0   
1  나폴리맛피아 콜라보, 감자연구소 콜라보 디저트 등\n재밌는 도전을 끊임없이 하는 롯...  19374      37.0   
2  🏍️ 최근 배달의민족 올해의 키워드로 ‘국밥’이 선정될 정도로 인기가 대단한데요.\...   4072      23.0   

                                        url 광고 미포함  \
0  https://www.instagram.com/p/DIs30jKzrXh/      O   
1  https://www.instagram.com/p/DIsOpVSzL0D/      O   
2  https://www.instagram.com/p/DIsH68vTAo1/      O   

                                            keywords  \
0  럭셔리, 스테이, 리조트, 아만 다리,

In [3]:
import pandas as pd
import ast
import os

def safe_eval_list(x):
    """
    데이터프레임의 키워드 컬럼을 리스트 객체로 안전하게 변환합니다.
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        # "[word1, word2]" 형태 처리
        if x.startswith('[') and x.endswith(']'):
            try:
                return ast.literal_eval(x)
            except:
                cleaned = x.strip('[]')
                return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        # "word1, word2" 형태 처리
        else:
            return [kw.strip() for kw in x.split(',') if kw.strip()]
    return []

def main():
    # 파일 경로 설정
    input_file = "/Users/hajiyoon/workspace/data/processed/knewnew_full_2025_v2.csv"
    output_file = "/Users/hajiyoon/workspace/data/processed/knewnew_full_2025_v2_filtered.csv"
    
    # 제거할 단어 리스트 (전달해주신 리스트를 기반으로 set 생성)
    words_to_remove = {
        "배달음식", "연령대별", "배달주문", "소비자결제", "주문순위", "배달앱", 
        "뮤직페스티벌", "파라다이스시티", "빗썸나눔", "엔터테인먼트", "매들리메들리", "GD", 
        "티머니", "결제", "모바일", "도입", "애플페이", "지하철", 
        "디올", "엘더플라워", "우디오크", "블랙베리", "1948년", "관능", "헤리티지", "자신감", "미스디올", "자스민플라워", 
        "디어블라썸커피", "마주이", "맨들러", "멜티도나", "잠실야구장", "하우스서울", "멘탈충전", "블랭크", "펠어커피초코", 
        "대학교", "하우스커피", "근황토크", "호말커피", "스승의날", "한옥카페", "교수님", 
        "고양이카페", "식빵", "츤데레", "카페유일", "상주냥", "하츠코히", 
        "스릴러", "공포", "외모지상주의", "젊음", "후유증", "서브스턴스"
    }

    if not os.path.exists(input_file):
        print(f"오류: 파일을 찾을 수 없습니다 -> {input_file}")
        return

    print(f"데이터 로드 중: {input_file}")
    df = pd.read_csv(input_file)
    
    if 'keywords_v2' not in df.columns:
        print("오류: CSV 파일에 'keywords_v2' 컬럼이 존재하지 않습니다.")
        return

    # 1. 키워드를 리스트로 변환하여 임시 컬럼 생성
    df['temp_list'] = df['keywords_v2'].apply(safe_eval_list)

    # 2. 지정된 단어 제거 작업
    print(f"키워드 필터링 중 (제거 대상: {len(words_to_remove)}종의 단어)...")
    
    # 리스트에서 해당 단어들을 제외하고 다시 쉼표로 연결된 문자열로 변환
    df['keywords_v2'] = df['temp_list'].apply(
        lambda kws: ", ".join([kw for kw in kws if kw not in words_to_remove])
    )

    # 임시 컬럼 삭제
    df = df.drop(columns=['temp_list'])

    # 3. 결과 저장
    try:
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"성공: 필터링된 데이터가 저장되었습니다.\n저장 경로: {output_file}")
        
        # 상위 결과 확인
        print("\n[변환된 키워드 샘플]")
        print(df['keywords_v2'].head())
    except Exception as e:
        print(f"오류: 파일 저장 중 문제가 발생했습니다: {e}")

if __name__ == "__main__":
    main()


데이터 로드 중: /Users/hajiyoon/workspace/data/processed/knewnew_full_2025_v2.csv
키워드 필터링 중 (제거 대상: 56종의 단어)...
성공: 필터링된 데이터가 저장되었습니다.
저장 경로: /Users/hajiyoon/workspace/data/processed/knewnew_full_2025_v2_filtered.csv

[변환된 키워드 샘플]
0    럭셔리, 스테이, 리조트, 아만다리, 호시노야가루이자와, 스테이폴리오, 인플루언서,...
1      롯데리아, 나폴리, 감자연구소, 디저트, 쥐포튀김, 떼리앙, 캐릭터, 띠부씰, 콜라보
2              국밥, 뉴웨이브, MZ세대, 힙함, 감성, 이색식재료, 플레이팅, 한끼
3        성수, 합정, 홍대, 을지로, 데이트, 카페, 핫플레이스, LP, 퇴근후, 아지트
4           오이, 소금빵, 디저트, 샌드위치, 바게트, 고리, 슬로우타운, 베이커리호프
Name: keywords_v2, dtype: str


In [6]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os
import pickle
import json
import random
import numpy as np

try:
    import community as community_louvain
except ImportError:
    os.system("pip install python-louvain")
    import community as community_louvain


# ============================================================
# 설정
# ============================================================

FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_full_2025_v2_filtered.csv"
COLUMN_NAME = "keywords_v2"
SAVE_DIR    = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"   # G, partition 저장 경로

G_PATH         = SAVE_DIR + "network_G.pkl"
PARTITION_PATH = SAVE_DIR + "partition.json"
RESULT_PATH    = SAVE_DIR + "community_centrality_results.csv"

RANDOM_SEED = 42  # 재현성 고정 시드


# ============================================================
# 유틸리티
# ============================================================

def safe_eval(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip("[]")
            return [kw.strip() for kw in cleaned.split(",") if kw.strip()]
        return []


# ============================================================
# Step 1. 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("[Step 1] 네트워크 생성 중...")
    df = pd.read_csv(file_path)
    df[column_name] = df[column_name].apply(safe_eval)
    print(f"  데이터 행 수: {len(df)}")

    # 고빈도 불용어
    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    high_freq_stopwords = {
        kw for kw, freq in kw_doc_freq.items()
        if freq >= total_docs * 0.30
    }
    manual_stopwords = {"뉴뉴", "뉴뉴매거진", "Knewnew", "Knewnewmagazine"}
    stopwords = high_freq_stopwords | manual_stopwords
    print(f"  불용어 제거: 총 {len(stopwords)}개")

    # 네트워크 생성
    G = nx.Graph()
    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]["weight"] += 1
            else:
                G.add_edge(u, v, weight=1)

    print(f"  노드: {G.number_of_nodes()}개 | 엣지: {G.number_of_edges()}개")
    return G


# ============================================================
# Step 2. Louvain 군집화 (재현성 보장)
# ============================================================

def detect_communities(G, seed=RANDOM_SEED):
    print("\n[Step 2] Louvain 커뮤니티 탐지 중...")

    # 전체 랜덤 시드 고정
    random.seed(seed)
    np.random.seed(seed)

    partition = community_louvain.best_partition(
        G, weight="weight", random_state=seed
    )
    community_sizes = Counter(partition.values())
    print(f"  발견된 군집 수: {len(community_sizes)}개")
    return partition


# ============================================================
# Step 3. 군집별 중심성 분석
# ============================================================

def analyze_communities(G, partition):
    print("\n[Step 3] 군집별 중심성 분석 중...")
    community_sizes = Counter(partition.values())
    results = []

    for comm_id, size in sorted(
        community_sizes.items(), key=lambda x: x[1], reverse=True
    ):
        members = [node for node, c in partition.items() if c == comm_id]
        sub_G   = G.subgraph(members).copy()

        # 군집 내부 중심성 계산
        deg = nx.degree_centrality(sub_G)

        try:
            eig = nx.eigenvector_centrality(
                sub_G, max_iter=1000, weight="weight"
            )
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}

        # betweenness: 가중치를 거리로 해석하므로 미적용
        bet = nx.betweenness_centrality(sub_G)

        top_deg = sorted(deg, key=deg.get, reverse=True)[:10]
        top_eig = sorted(eig, key=eig.get, reverse=True)[:10]
        top_bet = sorted(bet, key=bet.get, reverse=True)[:10]

        results.append({
            "군집ID":           comm_id,
            "키워드수":         size,
            "대세_Degree_Top10":    ", ".join(top_deg),
            "파워_Eigenvector_Top10": ", ".join(top_eig),
            "융합허브_Betweenness_Top10": ", ".join(top_bet),
        })

    result_df = pd.DataFrame(results)
    return result_df


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":

    # ── 재현성: 저장된 G, partition이 있으면 불러오고 없으면 새로 생성 ──
    if os.path.exists(G_PATH) and os.path.exists(PARTITION_PATH):
        print("[재현성 모드] 저장된 G, partition 불러오는 중...")
        with open(G_PATH, "rb") as f:
            G = pickle.load(f)
        with open(PARTITION_PATH, "r", encoding="utf-8") as f:
            partition = json.load(f)
        # json 저장 시 key가 문자열로 바뀌므로 정수 value는 그대로 유지
        partition = {k: v for k, v in partition.items()}
        print(f"  G 노드: {G.number_of_nodes()}개 | 엣지: {G.number_of_edges()}개")
        print(f"  군집 수: {len(set(partition.values()))}개")

    else:
        print("[신규 생성 모드] 네트워크와 군집을 새로 생성합니다.")
        G         = build_network(FILE_PATH, COLUMN_NAME)
        partition = detect_communities(G)

        # 결과 저장 (다음 실행부터 재현성 보장)
        with open(G_PATH, "wb") as f:
            pickle.dump(G, f)
        with open(PARTITION_PATH, "w", encoding="utf-8") as f:
            json.dump(partition, f, ensure_ascii=False)
        print(f"\n  G → '{G_PATH}' 저장 완료")
        print(f"  partition → '{PARTITION_PATH}' 저장 완료")

    # ── 군집별 중심성 분석 및 출력 ──
    result_df = analyze_communities(G, partition)
    result_df.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
    print(f"\n  결과 저장 → '{RESULT_PATH}'")

    # 터미널 출력
    pd.set_option("display.max_colwidth", 80)
    pd.set_option("display.max_rows", 50)
    print("\n[군집별 중심성 분석 결과]")
    print(result_df.to_string(index=False))

[신규 생성 모드] 네트워크와 군집을 새로 생성합니다.
[Step 1] 네트워크 생성 중...
  데이터 행 수: 1508
  불용어 제거: 총 4개
  노드: 5060개 | 엣지: 42487개

[Step 2] Louvain 커뮤니티 탐지 중...
  발견된 군집 수: 26개

  G → '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/network_G.pkl' 저장 완료
  partition → '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/partition.json' 저장 완료

[Step 3] 군집별 중심성 분석 중...

  결과 저장 → '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/community_centrality_results.csv'

[군집별 중심성 분석 결과]
 군집ID  키워드수                                            대세_Degree_Top10                                            파워_Eigenvector_Top10                                     융합허브_Betweenness_Top10
    1  1162                 카페, 디저트, 커피, 맛집, 브런치, F&B, 여행, 감성, 데이트, 힐링                      카페, 디저트, 맛집, F&B, 커피, 브런치, 여행, 데이트, 힐링, 감성                 카페, 디저트, F&B, 커피, 브런치, 맛집, 여행, 힐링, 감성, 데이트
    3   812         CU, 딸기, 말차, GS25, 스타벅스, 편의점, 헬로키티, 세븐일레븐, 콜라보, 오리온               CU, GS25, 말차, 세븐일레븐, 딸기, 편의점, 스타벅스, 오리온, 헬로키티, 망고     

In [7]:
import json
import pandas as pd
from collections import defaultdict

# ============================================================
# 설정
# ============================================================
SAVE_DIR       = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
ALL_NODES_PATH = SAVE_DIR + "community_all_keywords.csv"

# ✨ 네가 정리해준 군집명 딕셔너리
COMMUNITY_NAMES = {
    1: "감성 카페 라이프",
    3: "편의점 시즌 콜라보",
    7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트",
    9: "도쿄 미식 여행",
    13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집",
    14: "여름 보양·청량 푸드",
    8: "명절·연휴 간식",
    4: "건강식·저속노화",
    11: "유튜버 추천 간편식",
    5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수",
    0: "프리미엄 리빙·테이블웨어",
    19: "새해·시즌 베이커리",
    10: "로컬 카페 탐방",
    21: "프리미엄 핸드드립·오마카세",
    17: "자극적 한식·스트레스 해소食",
    15: "가을·겨울 패션 코디",
    16: "공연·피크닉 아웃도어",
    24: "마곡 로컬 카페",
    22: "이색 요리 실험",
    12: "빈티지 편집샵",
    25: "인플루언서 맛집",
    18: "일본식 선술집",
    23: "2025 상반기 결산"
}

def save_community_keywords_with_names():
    print("[작업 시작] 군집별 이름과 키워드 리스트를 추출할게...")

    # 1. partition.json 불러오기
    try:
        with open(PARTITION_PATH, "r", encoding="utf-8") as f:
            partition = json.load(f)
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없어: {PARTITION_PATH}")
        return

    # 2. 군집별로 노드 그룹화
    community_map = defaultdict(list)
    for node, comm_id in partition.items():
        comm_id = int(comm_id)
        community_map[comm_id].append(node)

    # 3. 데이터 리스트 생성
    result_data = []
    for comm_id, nodes in community_map.items():
        comm_name = COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}")
        
        result_data.append({
            "군집ID": comm_id,
            "군집명": comm_name,
            "키워드_개수": len(nodes),
            "전체_키워드": ", ".join(nodes)
        })

    # 4. 데이터프레임 생성 및 정렬
    df = pd.DataFrame(result_data)
    df = df[["군집ID", "군집명", "키워드_개수", "전체_키워드"]]
    df = df.sort_values(by="키워드_개수", ascending=False)

    # 5. CSV 저장
    df.to_csv(ALL_NODES_PATH, index=False, encoding="utf-8-sig")
    print(f"✅ 저장 완료: {ALL_NODES_PATH}")
    
    print("\n[추출 결과 미리보기]")
    print(df.head())

if __name__ == "__main__":
    save_community_keywords_with_names()

[작업 시작] 군집별 이름과 키워드 리스트를 추출할게...
✅ 저장 완료: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/community_all_keywords.csv

[추출 결과 미리보기]
   군집ID         군집명  키워드_개수  \
1     1   감성 카페 라이프    1162   
3     3  편의점 시즌 콜라보     812   
7     7  이자카야·바 다이닝     635   
6     6  선물·기념일 디저트     417   
9     9    도쿄 미식 여행     280   

                                                                            전체_키워드  
1  여행, 디저트, 감성, 합정, 홍대, 데이트, 카페, LP, 아지트, 오이, 소금빵, 샌드위치, 고리, 슬로우타운, 베이커리호프, 망미동...  
3  캐릭터, 콜라보, 이케아, 아이스크림, 꾸덕, 까르보나라, 마르게리따, 시나몬, 핫도그, 폴바셋, 밀도, 상하아이스크림, 스페셜티, 조합...  
7  퇴근후, 파스타, 포차, 와인, 한우, 제철식재료, 한우다이닝, 콜키지프리, 여의도, 황금연휴, 회식, 세운상가, 청계천, 도너츠, 맥주...  
6  핫플레이스, 청량, 식음료, 케이크, 선물, 이벤트, 음식, 영화, 문화생활, 미장센, 박찬욱, 봉준호, 팀버튼, 제주, 네일, 커플, ...  
9  먹방, 쇼핑, 도쿄, 서지수, 코이세이오, JOURNALSTANDARD, 브이로그, 스시, 마루미, 쿠시아키, 편집샵, 가성비, 실내데이...  


In [9]:
import os
import pickle
import json
import pandas as pd
import networkx as nx
from collections import Counter

# ============================================================
# 설정 (알려준 경로 그대로 적용)
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
RESULT_PATH    = SAVE_DIR + "뉴뉴_community_centrality_results.csv"

# ✨ 수정된 군집명 딕셔너리
COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 
    11: "미디어 푸드 콘텐츠", # 이름 변경됨
    5: "힙한 한끼 미식", 20: "수험생·이벤트 특수", 0: "프리미엄 리빙·테이블웨어",
    19: "새해·시즌 베이커리", 10: "로컬 카페 탐방", 21: "프리미엄 핸드드립·오마카세",
    17: "자극적 한식·스트레스 해소", # 이름 변경됨
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
    # 23번(2025 상반기 결산)은 딕셔너리에서 제외
}

def update_network_and_results():
    print("[작업 시작] 데이터를 업데이트합니다...")

    # 1. 기존 파일 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    # JSON 키-값을 올바른 타입으로 변환
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 23번 군집 노드 삭제
    TARGET_COMM_ID = 23
    nodes_to_remove = [node for node, comm_id in partition.items() if comm_id == TARGET_COMM_ID]
    
    # 그래프(G)에서 삭제
    G.remove_nodes_from(nodes_to_remove)
    # partition 딕셔너리에서 삭제
    for node in nodes_to_remove:
        del partition[node]
        
    print(f"  삭제된 노드 수 (군집 23): {len(nodes_to_remove)}개")

    # 3. 업데이트된 G와 partition 다시 저장 (덮어쓰기)
    with open(G_PATH, "wb") as f:
        pickle.dump(G, f)
    with open(PARTITION_PATH, "w", encoding="utf-8") as f:
        json.dump(partition, f, ensure_ascii=False)
    print("  ✅ G, partition 파일 업데이트 완료")

    # 4. 중심성 분석 재실행 (삭제된 상태 반영)
    print("  중심성 분석 다시 계산 중...")
    community_sizes = Counter(partition.values())
    results = []

    for comm_id, size in sorted(community_sizes.items(), key=lambda x: x[1], reverse=True):
        members = [node for node, c in partition.items() if c == comm_id]
        sub_G   = G.subgraph(members).copy()

        # 중심성 계산
        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}
        bet = nx.betweenness_centrality(sub_G)

        top_deg = sorted(deg, key=deg.get, reverse=True)[:10]
        top_eig = sorted(eig, key=eig.get, reverse=True)[:10]
        top_bet = sorted(bet, key=bet.get, reverse=True)[:10]

        # 결과 리스트에 추가 (군집명 포함)
        results.append({
            "군집ID":           comm_id,
            "군집명":           COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}"),
            "키워드수":         size,
            "대세_Degree_Top10":    ", ".join(top_deg),
            "파워_Eigenvector_Top10": ", ".join(top_eig),
            "융합허브_Betweenness_Top10": ", ".join(top_bet),
        })

    # 5. 결과 CSV 덮어쓰기
    result_df = pd.DataFrame(results)
    result_df.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
    print(f"  ✅ 중심성 분석 결과 업데이트 완료: {RESULT_PATH}")

if __name__ == "__main__":
    update_network_and_results()

[작업 시작] 데이터를 업데이트합니다...
  삭제된 노드 수 (군집 23): 4개
  ✅ G, partition 파일 업데이트 완료
  중심성 분석 다시 계산 중...
  ✅ 중심성 분석 결과 업데이트 완료: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_centrality_results.csv


In [11]:
import json
import pandas as pd
from collections import defaultdict

# ============================================================
# 설정
# ============================================================
SAVE_DIR       = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json" # 파일명 수정됨
ALL_NODES_PATH = SAVE_DIR + "뉴뉴_community_all_keywords.csv"

# ✨ 업데이트된 군집명 딕셔너리
COMMUNITY_NAMES = {
    1: "감성 카페 라이프",
    3: "편의점 시즌 콜라보",
    7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트",
    9: "도쿄 미식 여행",
    13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집",
    14: "여름 보양·청량 푸드",
    8: "명절·연휴 간식",
    4: "건강식·저속노화",
    11: "미디어 푸드 콘텐츠", # 이름 변경됨
    5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수",
    0: "프리미엄 리빙·테이블웨어",
    19: "새해·시즌 베이커리",
    10: "로컬 카페 탐방",
    21: "프리미엄 핸드드립·오마카세",
    17: "자극적 한식·스트레스 해소", # 이름 변경됨
    15: "가을·겨울 패션 코디",
    16: "공연·피크닉 아웃도어",
    24: "마곡 로컬 카페",
    22: "이색 요리 실험",
    12: "빈티지 편집샵",
    25: "인플루언서 맛집",
    18: "일본식 선술집"
    # 23번은 딕셔너리에서도 삭제
}

def save_community_keywords_with_names():
    print("[작업 시작] 군집별 이름과 키워드 리스트를 추출할게...")

    # 1. partition.json 불러오기
    try:
        with open(PARTITION_PATH, "r", encoding="utf-8") as f:
            partition = json.load(f)
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없어: {PARTITION_PATH}")
        return

    # 2. 군집별로 노드 그룹화
    community_map = defaultdict(list)
    for node, comm_id in partition.items():
        comm_id = int(comm_id)
        # ✨ 23번 군집은 여기서 아예 빼버리기 (안전장치)
        if comm_id == 23:
            continue
        community_map[comm_id].append(node)

    # 3. 데이터 리스트 생성
    result_data = []
    for comm_id, nodes in community_map.items():
        comm_name = COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}")
        
        result_data.append({
            "군집ID": comm_id,
            "군집명": comm_name,
            "키워드_개수": len(nodes),
            "전체_키워드": ", ".join(nodes)
        })

    # 4. 데이터프레임 생성 및 정렬
    df = pd.DataFrame(result_data)
    df = df[["군집ID", "군집명", "키워드_개수", "전체_키워드"]]
    df = df.sort_values(by="키워드_개수", ascending=False)

    # 5. CSV 저장
    df.to_csv(ALL_NODES_PATH, index=False, encoding="utf-8-sig")
    print(f"✅ 저장 완료: {ALL_NODES_PATH}")
    
    print("\n[추출 결과 미리보기]")
    print(df.head())

if __name__ == "__main__":
    save_community_keywords_with_names()

[작업 시작] 군집별 이름과 키워드 리스트를 추출할게...
✅ 저장 완료: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords.csv

[추출 결과 미리보기]
   군집ID         군집명  키워드_개수  \
1     1   감성 카페 라이프    1162   
3     3  편의점 시즌 콜라보     812   
7     7  이자카야·바 다이닝     635   
6     6  선물·기념일 디저트     417   
9     9    도쿄 미식 여행     280   

                                                                            전체_키워드  
1  여행, 디저트, 감성, 합정, 홍대, 데이트, 카페, LP, 아지트, 오이, 소금빵, 샌드위치, 고리, 슬로우타운, 베이커리호프, 망미동...  
3  캐릭터, 콜라보, 이케아, 아이스크림, 꾸덕, 까르보나라, 마르게리따, 시나몬, 핫도그, 폴바셋, 밀도, 상하아이스크림, 스페셜티, 조합...  
7  퇴근후, 파스타, 포차, 와인, 한우, 제철식재료, 한우다이닝, 콜키지프리, 여의도, 황금연휴, 회식, 세운상가, 청계천, 도너츠, 맥주...  
6  핫플레이스, 청량, 식음료, 케이크, 선물, 이벤트, 음식, 영화, 문화생활, 미장센, 박찬욱, 봉준호, 팀버튼, 제주, 네일, 커플, ...  
9  먹방, 쇼핑, 도쿄, 서지수, 코이세이오, JOURNALSTANDARD, 브이로그, 스시, 마루미, 쿠시아키, 편집샵, 가성비, 실내데이...  


In [12]:
import os
import pickle
import json
import pandas as pd
import networkx as nx
import numpy as np

# ============================================================
# 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
TEST_RESULT_PATH = SAVE_DIR + "뉴뉴_filtering_candidates_test.csv" # 눈으로 확인할 파일

# 군집명 딕셔너리
COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 11: "미디어 푸드 콘텐츠", 5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수", 0: "프리미엄 리빙·테이블웨어", 19: "새해·시즌 베이커리",
    10: "로컬 카페 탐방", 21: "프리미엄 핸드드립·오마카세", 17: "자극적 한식·스트레스 해소",
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
}

def find_filtering_candidates():
    print("[작업 시작] 하위 10% 필터링 대상을 찾습니다 (원본 유지)...")

    # 1. 파일 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 군집별로 하위 10% 노드 찾기
    communities = set(partition.values())
    result_data = []
    total_removed = 0

    for comm_id in communities:
        nodes = [n for n, c in partition.items() if c == comm_id]
        if len(nodes) < 3: # 노드가 너무 적은 군집은 패스
            continue
            
        sub_G = G.subgraph(nodes).copy()

        # 세 가지 중심성 계산
        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}
        bet = nx.betweenness_centrality(sub_G)

        # 하위 10% 기준값(Threshold) 계산
        deg_th = np.percentile(list(deg.values()), 10)
        eig_th = np.percentile(list(eig.values()), 10)
        bet_th = np.percentile(list(bet.values()), 10)

        # 조건: 세 지표 모두 기준값 이하인 교집합 노드 찾기
        candidates = []
        for n in nodes:
            if deg[n] <= deg_th and eig[n] <= eig_th and bet[n] <= bet_th:
                candidates.append(n)

        # 결과 저장
        comm_name = COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}")
        total_removed += len(candidates)
        
        result_data.append({
            "군집ID": comm_id,
            "군집명": comm_name,
            "기존_키워드수": len(nodes),
            "삭제예상_키워드수": len(candidates),
            "삭제예상_키워드목록": ", ".join(candidates) if candidates else "없음"
        })

    # 3. 데이터프레임 저장 및 출력
    df = pd.DataFrame(result_data)
    df = df.sort_values(by="삭제예상_키워드수", ascending=False)
    
    df.to_csv(TEST_RESULT_PATH, index=False, encoding="utf-8-sig")
    
    print(f"\n✅ 전체 데이터에서 총 {total_removed}개의 키워드가 필터링될 예정이야!")
    print(f"✅ 상세 목록이 '{TEST_RESULT_PATH}'에 저장됐어.")
    print("\n[삭제 예상 상위 5개 군집 미리보기]")
    print(df[["군집명", "삭제예상_키워드수", "삭제예상_키워드목록"]].head())

if __name__ == "__main__":
    find_filtering_candidates()

[작업 시작] 하위 10% 필터링 대상을 찾습니다 (원본 유지)...

✅ 전체 데이터에서 총 201개의 키워드가 필터링될 예정이야!
✅ 상세 목록이 '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_filtering_candidates_test.csv'에 저장됐어.

[삭제 예상 상위 5개 군집 미리보기]
          군집명  삭제예상_키워드수  \
3  편의점 시즌 콜라보         45   
7  이자카야·바 다이닝         28   
1   감성 카페 라이프         25   
9    도쿄 미식 여행         14   
6  선물·기념일 디저트         14   

                                                                        삭제예상_키워드목록  
3  도트블랭킷, 이라선, 북촌돌담길, 국제갤러리, 활력, 코어, 요거트퍼플, 월드투어, IP, 고양, 글렌피딕, 런닝, 요가, 모히또, 고수...  
7  피서, 솥뚜껑, 스와이시, 밀양, 닭날개, 계온, 냉장육, 즉흥, 수족냉증, 글러브, 장갑, KnewnewSeries, 건강식단, 클린푸...  
1  브런치바, 속초명물, 나이트프루티, 성숙함, 물멍, 해변, 디제이, 연장, 문구점, 동궁과월지, 맘모스, 튀김소보로, 말차크림, 단쌉, ...  
9  F&B브랜드, 관광객, 트렌디한동네, BIFF, 영화제, 간단식사, 하루의마무리, 이재모피자, 패왕차희, 찻잎, 콜건적, 제로콜라, 코카...  
6                    영감, 우후, 아이만, 기술, 포도, 석류, 감, 서부, 카우보이, 건식, 생선정육점, 포장, 귤까기, 귤아트  


In [13]:
import os
import pickle
import json
import pandas as pd
import networkx as nx
import numpy as np

# ============================================================
# 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
TEST_RESULT_PATH = SAVE_DIR + "뉴뉴_filtering_candidates_test_5%.csv" # 눈으로 확인할 파일

# ✨ 군집명 딕셔너리
COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 11: "미디어 푸드 콘텐츠", 5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수", 0: "프리미엄 리빙·테이블웨어", 19: "새해·시즌 베이커리",
    10: "로컬 카페 탐방", 21: "프리미엄 핸드드립·오마카세", 17: "자극적 한식·스트레스 해소",
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
}

def find_filtering_candidates_5_percent():
    print("[작업 시작] 하위 5% 필터링 대상을 찾습니다 (원본 유지)...")

    # 1. 파일 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 군집별로 하위 5% 노드 찾기
    communities = set(partition.values())
    result_data = []
    total_removed = 0

    for comm_id in communities:
        nodes = [n for n, c in partition.items() if c == comm_id]
        if len(nodes) < 3: # 노드가 너무 적은 군집은 패스
            continue
            
        sub_G = G.subgraph(nodes).copy()

        # 세 가지 중심성 계산
        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}
        bet = nx.betweenness_centrality(sub_G)

        # ✨ 하위 5% 기준값(Threshold) 계산으로 수정
        deg_th = np.percentile(list(deg.values()), 5)
        eig_th = np.percentile(list(eig.values()), 5)
        bet_th = np.percentile(list(bet.values()), 5)

        # 조건: 세 지표 모두 하위 5% 이하인 교집합 노드 찾기
        candidates = []
        for n in nodes:
            if deg[n] <= deg_th and eig[n] <= eig_th and bet[n] <= bet_th:
                candidates.append(n)

        # 결과 저장
        comm_name = COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}")
        total_removed += len(candidates)
        
        result_data.append({
            "군집ID": comm_id,
            "군집명": comm_name,
            "기존_키워드수": len(nodes),
            "삭제예상_키워드수": len(candidates),
            "삭제예상_키워드목록": ", ".join(candidates) if candidates else "없음"
        })

    # 3. 데이터프레임 저장 및 출력
    df = pd.DataFrame(result_data)
    df = df.sort_values(by="삭제예상_키워드수", ascending=False)
    
    df.to_csv(TEST_RESULT_PATH, index=False, encoding="utf-8-sig")
    
    print(f"\n✅ 전체 데이터에서 총 {total_removed}개의 키워드가 필터링될 예정이야!")
    print(f"✅ 상세 목록이 '{TEST_RESULT_PATH}'에 저장됐어.")
    print("\n[삭제 예상 상위 5개 군집 미리보기]")
    print(df[["군집명", "삭제예상_키워드수", "삭제예상_키워드목록"]].head())

if __name__ == "__main__":
    find_filtering_candidates_5_percent()

[작업 시작] 하위 5% 필터링 대상을 찾습니다 (원본 유지)...

✅ 전체 데이터에서 총 99개의 키워드가 필터링될 예정이야!
✅ 상세 목록이 '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_filtering_candidates_test_5%.csv'에 저장됐어.

[삭제 예상 상위 5개 군집 미리보기]
           군집명  삭제예상_키워드수  \
1    감성 카페 라이프         14   
3   편의점 시즌 콜라보         13   
12     빈티지 편집샵          8   
18     일본식 선술집          7   
24    인플루언서 맛집          7   

                                                                        삭제예상_키워드목록  
1   브런치바, 나이트프루티, 물멍, 해변, 문구점, 동궁과월지, 맘모스, 튀김소보로, 말차크림, 단쌉, 오픈, 한국맥도날드, 계절음식, 과일시루  
3                 활력, 코어, 요거트퍼플, 내한, 스캇, 김장철, 김장, 배추, 신선도, 멜로우스트리트, 수타면, 라이즈, 소주안주  
12                                   롱드라이버스, 몬나니, 무어, 피클스, 츠오흐커피, 오토토빈티지, 포스터바, 니사  
18                                          선술집, 타치노미, 서서, 가벼운식사, 스탠딩바, 미츠바, 키보.co  
24                                             걍밍경, 차밥열끼, 효은옥, 육개옥, 미나미, 상아김밥, 타카이  


In [ ]:
#-----3개의 중심성 지표 모두 하위 10%인 노드 찾는 코드 (원본 유지)-----
import os
import pickle
import json
import pandas as pd
import networkx as nx
import numpy as np
from collections import Counter

# ============================================================
# 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
RESULT_PATH    = SAVE_DIR + "뉴뉴_community_centrality_results.csv"

COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 11: "미디어 푸드 콘텐츠", 5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수", 0: "프리미엄 리빙·테이블웨어", 19: "새해·시즌 베이커리",
    10: "로컬 카페 탐방", 21: "프리미엄 핸드드립·오마카세", 17: "자극적 한식·스트레스 해소",
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
}

# ✨ 절대 지우면 안 되는 키워드 목록
SAVE_KEYWORDS = {
    "모히또", "고수모히또", "생초콜릿", "앙금", "크림초콜릿", "컵파스타", 
    "요거트퍼플", "핫소스", "로제불닭", "까르보불닭", "발효음식", "클린푸드", 
    "말차크림", "단쌉", "과일시루", "튀김소보로", "맘모스", "계절음식", 
    "이재모피자", "간단식사", "걍밍경", "차밥열끼", "효은옥", "육개옥", 
    "미나미", "상아김밥", "타카이", "먹부림", "애프터눈티", "양념게장", 
    "밥도둑", "식욕"
}

def apply_10_percent_filter_with_exceptions():
    print("[작업 시작] 10% 필터링 적용 및 중요 키워드 보호 중...")

    # 1. 파일 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 삭제할 노드 찾기
    communities = set(partition.values())
    nodes_to_remove = []

    for comm_id in communities:
        nodes = [n for n, c in partition.items() if c == comm_id]
        if len(nodes) < 3:
            continue
            
        sub_G = G.subgraph(nodes).copy()

        # 중심성 계산
        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}
        bet = nx.betweenness_centrality(sub_G)

        # 10% 기준값 계산
        deg_th = np.percentile(list(deg.values()), 10)
        eig_th = np.percentile(list(eig.values()), 10)
        bet_th = np.percentile(list(bet.values()), 10)

        # ✨ 삭제 조건 + 예외 처리
        for n in nodes:
            if deg[n] <= deg_th and eig[n] <= eig_th and bet[n] <= bet_th:
                if n not in SAVE_KEYWORDS:  # 살릴 키워드가 아닐 때만 삭제 리스트에 추가
                    nodes_to_remove.append(n)

    # 3. 데이터에서 실제로 노드 삭제
    G.remove_nodes_from(nodes_to_remove)
    for n in nodes_to_remove:
        del partition[n]
        
    print(f"  ✅ 총 {len(nodes_to_remove)}개의 쓸모없는 키워드가 삭제됐어!")

    # 4. G, partition 파일 덮어쓰기
    with open(G_PATH, "wb") as f:
        pickle.dump(G, f)
    with open(PARTITION_PATH, "w", encoding="utf-8") as f:
        json.dump(partition, f, ensure_ascii=False)

    # 5. 삭제된 상태로 중심성 분석 재실행 및 결과 덮어쓰기
    print("  새로운 네트워크로 중심성 분석 재계산 중...")
    community_sizes = Counter(partition.values())
    results = []

    for comm_id, size in sorted(community_sizes.items(), key=lambda x: x[1], reverse=True):
        members = [node for node, c in partition.items() if c == comm_id]
        sub_G   = G.subgraph(members).copy()

        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}
        bet = nx.betweenness_centrality(sub_G)

        top_deg = sorted(deg, key=deg.get, reverse=True)[:10]
        top_eig = sorted(eig, key=eig.get, reverse=True)[:10]
        top_bet = sorted(bet, key=bet.get, reverse=True)[:10]

        results.append({
            "군집ID":           comm_id,
            "군집명":           COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}"),
            "키워드수":         size,
            "대세_Degree_Top10":    ", ".join(top_deg),
            "파워_Eigenvector_Top10": ", ".join(top_eig),
            "융합허브_Betweenness_Top10": ", ".join(top_bet),
        })

    result_df = pd.DataFrame(results)
    result_df.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
    print(f"  ✅ 모든 파일 업데이트 완료!")

if __name__ == "__main__":
    apply_10_percent_filter_with_exceptions()

[작업 시작] 10% 필터링 적용 및 중요 키워드 보호 중...
  ✅ 총 169개의 쓸모없는 키워드가 삭제됐어!
  새로운 네트워크로 중심성 분석 재계산 중...
  ✅ 모든 파일 업데이트 완료!


In [15]:
import json
import pandas as pd
from collections import defaultdict

# ============================================================
# 설정
# ============================================================
SAVE_DIR       = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json" # 파일명 수정됨
ALL_NODES_PATH = SAVE_DIR + "뉴뉴_community_all_keywords_v2.csv"

# ✨ 업데이트된 군집명 딕셔너리
COMMUNITY_NAMES = {
    1: "감성 카페 라이프",
    3: "편의점 시즌 콜라보",
    7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트",
    9: "도쿄 미식 여행",
    13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집",
    14: "여름 보양·청량 푸드",
    8: "명절·연휴 간식",
    4: "건강식·저속노화",
    11: "미디어 푸드 콘텐츠", # 이름 변경됨
    5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수",
    0: "프리미엄 리빙·테이블웨어",
    19: "새해·시즌 베이커리",
    10: "로컬 카페 탐방",
    21: "프리미엄 핸드드립·오마카세",
    17: "자극적 한식·스트레스 해소", # 이름 변경됨
    15: "가을·겨울 패션 코디",
    16: "공연·피크닉 아웃도어",
    24: "마곡 로컬 카페",
    22: "이색 요리 실험",
    12: "빈티지 편집샵",
    25: "인플루언서 맛집",
    18: "일본식 선술집"
    # 23번은 딕셔너리에서도 삭제
}

def save_community_keywords_with_names():
    print("[작업 시작] 군집별 이름과 키워드 리스트를 추출할게...")

    # 1. partition.json 불러오기
    try:
        with open(PARTITION_PATH, "r", encoding="utf-8") as f:
            partition = json.load(f)
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없어: {PARTITION_PATH}")
        return

    # 2. 군집별로 노드 그룹화
    community_map = defaultdict(list)
    for node, comm_id in partition.items():
        comm_id = int(comm_id)
        # ✨ 23번 군집은 여기서 아예 빼버리기 (안전장치)
        if comm_id == 23:
            continue
        community_map[comm_id].append(node)

    # 3. 데이터 리스트 생성
    result_data = []
    for comm_id, nodes in community_map.items():
        comm_name = COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}")
        
        result_data.append({
            "군집ID": comm_id,
            "군집명": comm_name,
            "키워드_개수": len(nodes),
            "전체_키워드": ", ".join(nodes)
        })

    # 4. 데이터프레임 생성 및 정렬
    df = pd.DataFrame(result_data)
    df = df[["군집ID", "군집명", "키워드_개수", "전체_키워드"]]
    df = df.sort_values(by="키워드_개수", ascending=False)

    # 5. CSV 저장
    df.to_csv(ALL_NODES_PATH, index=False, encoding="utf-8-sig")
    print(f"✅ 저장 완료: {ALL_NODES_PATH}")
    
    print("\n[추출 결과 미리보기]")
    print(df.head())

if __name__ == "__main__":
    save_community_keywords_with_names()

[작업 시작] 군집별 이름과 키워드 리스트를 추출할게...
✅ 저장 완료: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords_v2.csv

[추출 결과 미리보기]
   군집ID         군집명  키워드_개수  \
1     1   감성 카페 라이프    1143   
3     3  편의점 시즌 콜라보     774   
7     7  이자카야·바 다이닝     612   
6     6  선물·기념일 디저트     403   
9     9    도쿄 미식 여행     268   

                                                                            전체_키워드  
1  여행, 디저트, 감성, 합정, 홍대, 데이트, 카페, LP, 아지트, 오이, 소금빵, 샌드위치, 고리, 슬로우타운, 베이커리호프, 망미동...  
3  캐릭터, 콜라보, 이케아, 아이스크림, 꾸덕, 까르보나라, 마르게리따, 시나몬, 핫도그, 폴바셋, 밀도, 상하아이스크림, 스페셜티, 조합...  
7  퇴근후, 파스타, 포차, 와인, 한우, 제철식재료, 한우다이닝, 콜키지프리, 여의도, 황금연휴, 회식, 세운상가, 청계천, 도너츠, 맥주...  
6  핫플레이스, 청량, 식음료, 케이크, 선물, 이벤트, 음식, 영화, 문화생활, 미장센, 박찬욱, 봉준호, 팀버튼, 제주, 네일, 커플, ...  
9  먹방, 쇼핑, 도쿄, 서지수, 코이세이오, JOURNALSTANDARD, 브이로그, 스시, 마루미, 쿠시아키, 편집샵, 가성비, 실내데이...  


In [18]:
import os
import pickle
import json
import networkx as nx
from pyvis.network import Network

# ============================================================
# 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
HTML_SAVE_DIR  = SAVE_DIR + "다시!!visualizations_html/"

if not os.path.exists(HTML_SAVE_DIR):
    os.makedirs(HTML_SAVE_DIR)

COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 11: "미디어 푸드 콘텐츠", 5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수", 0: "프리미엄 리빙·테이블웨어", 19: "새해·시즌 베이커리",
    10: "로컬 카페 탐방", 21: "프리미엄 핸드드립·오마카세", 17: "자극적 한식·스트레스 해소",
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
}

def draw_fixed_html_networks():
    print("[작업 시작] 군집별 고정형 HTML 네트워크를 생성합니다...")

    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    for comm_id in sorted(set(partition.values())):
        nodes = [n for n, c in partition.items() if c == comm_id]
        if not nodes: continue
        
        sub_G = G.subgraph(nodes).copy()
        comm_name = COMMUNITY_NAMES.get(comm_id, f"Cluster_{comm_id}")
        
        print(f"  - '{comm_name}' HTML 생성 중...")

        # Pyvis 객체 생성
        net = Network(height="900px", width="100%", bgcolor="#ffffff", font_color="black", directed=False)
        net.from_nx(sub_G)

        # 노드/엣지 스타일 미세 조정
        degrees = dict(sub_G.degree())
        for node in net.nodes:
            node_id = node['id']
            node['size'] = degrees.get(node_id, 1) * 2 + 10
            node['font'] = {'size': 14, 'face': 'Arial'}

        # ✨ 파이썬 문법에 맞게 True로 수정 (첫 글자 대문자!)
        options = {
          "nodes": { "scaling": { "min": 10, "max": 50 } },
          "edges": { "smooth": { "type": "continuous" } },
          "physics": {
            "enabled": False,
            "barnesHut": {
              "gravitationalConstant": -80000,
              "avoidOverlap": 1
            }
          },
          "interaction": { 
              "hover": True,            # 👈 true를 True로 수정
              "navigationButtons": True # 👈 true를 True로 수정
          }
        }
        
        # 딕셔너리를 JSON 문자열로 변환하여 적용
        net.set_options(json.dumps(options))

        # 파일 저장
        file_name = f"community_{comm_id}_{comm_name.replace(' ', '_')}.html"
        save_path = os.path.join(HTML_SAVE_DIR, file_name)
        
        net.save_graph(save_path)

    print(f"\n✅ 시각화 완료! '{HTML_SAVE_DIR}' 폴더를 확인해봐!")

if __name__ == "__main__":
    draw_fixed_html_networks()

[작업 시작] 군집별 고정형 HTML 네트워크를 생성합니다...
  - '프리미엄 리빙·테이블웨어' HTML 생성 중...
  - '감성 카페 라이프' HTML 생성 중...
  - '성수·을지로 로컬 맛집' HTML 생성 중...
  - '편의점 시즌 콜라보' HTML 생성 중...
  - '건강식·저속노화' HTML 생성 중...
  - '힙한 한끼 미식' HTML 생성 중...
  - '선물·기념일 디저트' HTML 생성 중...
  - '이자카야·바 다이닝' HTML 생성 중...
  - '명절·연휴 간식' HTML 생성 중...
  - '도쿄 미식 여행' HTML 생성 중...
  - '로컬 카페 탐방' HTML 생성 중...
  - '미디어 푸드 콘텐츠' HTML 생성 중...
  - '연말 파인다이닝' HTML 생성 중...
  - '여름 보양·청량 푸드' HTML 생성 중...
  - '가을·겨울 패션 코디' HTML 생성 중...
  - '공연·피크닉 아웃도어' HTML 생성 중...
  - '자극적 한식·스트레스 해소' HTML 생성 중...
  - '새해·시즌 베이커리' HTML 생성 중...
  - '수험생·이벤트 특수' HTML 생성 중...
  - '프리미엄 핸드드립·오마카세' HTML 생성 중...
  - '이색 요리 실험' HTML 생성 중...
  - '마곡 로컬 카페' HTML 생성 중...
  - '인플루언서 맛집' HTML 생성 중...

✅ 시각화 완료! '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/다시!!visualizations_html/' 폴더를 확인해봐!


In [19]:
#--- 프리미엄 리빙 군집 삭제 코드----
import os
import pickle
import json
import pandas as pd
import networkx as nx
from collections import Counter

# ============================================================
# 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
RESULT_PATH    = SAVE_DIR + "뉴뉴_community_centrality_results.csv"

# ✨ 0번(프리미엄 리빙·테이블웨어) 군집은 딕셔너리에서도 뺐어!
COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 11: "미디어 푸드 콘텐츠", 5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수", 19: "새해·시즌 베이커리", 10: "로컬 카페 탐방", 
    21: "프리미엄 핸드드립·오마카세", 17: "자극적 한식·스트레스 해소",
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
}

def remove_community_and_update():
    print("[작업 시작] '프리미엄 리빙·테이블웨어' 군집을 삭제합니다...")

    # 1. 기존 파일 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 0번 군집(프리미엄 리빙·테이블웨어) 노드 삭제
    TARGET_COMM_ID = 0
    nodes_to_remove = [node for node, comm_id in partition.items() if comm_id == TARGET_COMM_ID]
    
    # 그래프(G)와 partition에서 완전히 지우기
    G.remove_nodes_from(nodes_to_remove)
    for node in nodes_to_remove:
        del partition[node]
        
    print(f"  ✅ 삭제된 노드 수 (0번 군집): {len(nodes_to_remove)}개")

    # 3. 업데이트된 G와 partition 다시 저장
    with open(G_PATH, "wb") as f:
        pickle.dump(G, f)
    with open(PARTITION_PATH, "w", encoding="utf-8") as f:
        json.dump(partition, f, ensure_ascii=False)

    # 4. 중심성 분석 재실행 (삭제된 상태 반영)
    print("  새로운 네트워크로 중심성 분석 다시 계산 중...")
    community_sizes = Counter(partition.values())
    results = []

    for comm_id, size in sorted(community_sizes.items(), key=lambda x: x[1], reverse=True):
        members = [node for node, c in partition.items() if c == comm_id]
        sub_G   = G.subgraph(members).copy()

        # 중심성 계산
        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}
        bet = nx.betweenness_centrality(sub_G)

        top_deg = sorted(deg, key=deg.get, reverse=True)[:10]
        top_eig = sorted(eig, key=eig.get, reverse=True)[:10]
        top_bet = sorted(bet, key=bet.get, reverse=True)[:10]

        results.append({
            "군집ID":           comm_id,
            "군집명":           COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}"),
            "키워드수":         size,
            "대세_Degree_Top10":    ", ".join(top_deg),
            "파워_Eigenvector_Top10": ", ".join(top_eig),
            "융합허브_Betweenness_Top10": ", ".join(top_bet),
        })

    # 5. 결과 CSV 덮어쓰기
    result_df = pd.DataFrame(results)
    result_df.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
    print(f"  ✅ 모든 파일 업데이트 완료: {RESULT_PATH}")

if __name__ == "__main__":
    remove_community_and_update()

[작업 시작] '프리미엄 리빙·테이블웨어' 군집을 삭제합니다...
  ✅ 삭제된 노드 수 (0번 군집): 93개
  새로운 네트워크로 중심성 분석 다시 계산 중...
  ✅ 모든 파일 업데이트 완료: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_centrality_results.csv


In [20]:
import json
import pandas as pd
from collections import defaultdict

# ============================================================
# 설정
# ============================================================
SAVE_DIR       = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json" # 파일명 수정됨
ALL_NODES_PATH = SAVE_DIR + "뉴뉴_community_all_keywords_v2.csv"

# ✨ 업데이트된 군집명 딕셔너리
COMMUNITY_NAMES = {
    1: "감성 카페 라이프",
    3: "편의점 시즌 콜라보",
    7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트",
    9: "도쿄 미식 여행",
    13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집",
    14: "여름 보양·청량 푸드",
    8: "명절·연휴 간식",
    4: "건강식·저속노화",
    11: "미디어 푸드 콘텐츠", # 이름 변경됨
    5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수",
    0: "프리미엄 리빙·테이블웨어",
    19: "새해·시즌 베이커리",
    10: "로컬 카페 탐방",
    21: "프리미엄 핸드드립·오마카세",
    17: "자극적 한식·스트레스 해소", # 이름 변경됨
    15: "가을·겨울 패션 코디",
    16: "공연·피크닉 아웃도어",
    24: "마곡 로컬 카페",
    22: "이색 요리 실험",
    12: "빈티지 편집샵",
    25: "인플루언서 맛집",
    18: "일본식 선술집"
    # 23번은 딕셔너리에서도 삭제
}

def save_community_keywords_with_names():
    print("[작업 시작] 군집별 이름과 키워드 리스트를 추출할게...")

    # 1. partition.json 불러오기
    try:
        with open(PARTITION_PATH, "r", encoding="utf-8") as f:
            partition = json.load(f)
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없어: {PARTITION_PATH}")
        return

    # 2. 군집별로 노드 그룹화
    community_map = defaultdict(list)
    for node, comm_id in partition.items():
        comm_id = int(comm_id)
        # ✨ 23번 군집은 여기서 아예 빼버리기 (안전장치)
        if comm_id == 23:
            continue
        community_map[comm_id].append(node)

    # 3. 데이터 리스트 생성
    result_data = []
    for comm_id, nodes in community_map.items():
        comm_name = COMMUNITY_NAMES.get(comm_id, f"미정_군집_{comm_id}")
        
        result_data.append({
            "군집ID": comm_id,
            "군집명": comm_name,
            "키워드_개수": len(nodes),
            "전체_키워드": ", ".join(nodes)
        })

    # 4. 데이터프레임 생성 및 정렬
    df = pd.DataFrame(result_data)
    df = df[["군집ID", "군집명", "키워드_개수", "전체_키워드"]]
    df = df.sort_values(by="키워드_개수", ascending=False)

    # 5. CSV 저장
    df.to_csv(ALL_NODES_PATH, index=False, encoding="utf-8-sig")
    print(f"✅ 저장 완료: {ALL_NODES_PATH}")
    
    print("\n[추출 결과 미리보기]")
    print(df.head())

if __name__ == "__main__":
    save_community_keywords_with_names()

[작업 시작] 군집별 이름과 키워드 리스트를 추출할게...
✅ 저장 완료: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords_v2.csv

[추출 결과 미리보기]
   군집ID         군집명  키워드_개수  \
0     1   감성 카페 라이프    1143   
2     3  편의점 시즌 콜라보     774   
6     7  이자카야·바 다이닝     612   
5     6  선물·기념일 디저트     403   
8     9    도쿄 미식 여행     268   

                                                                            전체_키워드  
0  여행, 디저트, 감성, 합정, 홍대, 데이트, 카페, LP, 아지트, 오이, 소금빵, 샌드위치, 고리, 슬로우타운, 베이커리호프, 망미동...  
2  캐릭터, 콜라보, 이케아, 아이스크림, 꾸덕, 까르보나라, 마르게리따, 시나몬, 핫도그, 폴바셋, 밀도, 상하아이스크림, 스페셜티, 조합...  
6  퇴근후, 파스타, 포차, 와인, 한우, 제철식재료, 한우다이닝, 콜키지프리, 여의도, 황금연휴, 회식, 세운상가, 청계천, 도너츠, 맥주...  
5  핫플레이스, 청량, 식음료, 케이크, 선물, 이벤트, 음식, 영화, 문화생활, 미장센, 박찬욱, 봉준호, 팀버튼, 제주, 네일, 커플, ...  
8  먹방, 쇼핑, 도쿄, 서지수, 코이세이오, JOURNALSTANDARD, 브이로그, 스시, 마루미, 쿠시아키, 편집샵, 가성비, 실내데이...  


In [24]:
import os
import pickle
import json
import networkx as nx
from pyvis.network import Network
from collections import Counter

# ============================================================
# 1. 파일 경로 및 설정 
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
HTML_SAVE_DIR  = SAVE_DIR + "최종!!!community_networks_advanced/"

os.makedirs(HTML_SAVE_DIR, exist_ok=True)

COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 11: "미디어 푸드 콘텐츠", 5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수", 19: "새해·시즌 베이커리", 10: "로컬 카페 탐방", 
    21: "프리미엄 핸드드립·오마카세", 17: "자극적 한식·스트레스 해소",
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
}

PALETTE = [
    "#FF5A5F", "#087E8B", "#3C3C3C", "#F58231", "#911EB4", 
    "#46F0F0", "#F032E6", "#BCF60C", "#FABED4", "#008080",
    "#E6194B", "#3CB44B", "#FFE119", "#4363D8", "#F58231"
]

def draw_static_networks():
    print("[작업 시작] 춤추지 않는 완전 고정형 네트워크를 그립니다...")

    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    community_sizes = Counter(partition.values())

    for comm_id, size in sorted(community_sizes.items(), key=lambda x: x[1], reverse=True):
        if comm_id not in COMMUNITY_NAMES:
            continue
            
        megatrend = COMMUNITY_NAMES.get(comm_id, f"군집_{comm_id}")
        members   = [node for node, c in partition.items() if c == comm_id]
        sub_G     = G.subgraph(members).copy()

        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}

        eig_vals = list(eig.values())
        eig_min, eig_max = min(eig_vals) if eig_vals else 0, max(eig_vals) if eig_vals else 0

        def eig_to_border(val):
            return 1 + ((val - eig_min) / (eig_max - eig_min + 1e-9)) * 9

        color = PALETTE[list(sorted(community_sizes, key=community_sizes.get, reverse=True)).index(comm_id) % len(PALETTE)]

        net = Network(height="800px", width="100%", bgcolor="#1a1a2e", font_color="#ffffff", notebook=False)
        
        # ✨ 핵심 1: HTML의 물리 엔진(Physics)을 아예 처음부터 꺼버려! (enabled: false)
        net.set_options("""
        {
          "physics": { "enabled": false },
          "nodes": {
            "shape": "dot",
            "font": {"size": 13, "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"},
            "borderWidthSelected": 4
          },
          "edges": {"smooth": {"enabled": true, "type": "continuous"}},
          "interaction": {
            "hover": true, "tooltipDelay": 100, "hideEdgesOnDrag": true,
            "navigationButtons": true, "keyboard": true
          }
        }
        """)

        # ✨ 핵심 2: 파이썬(NetworkX)을 이용해 미리 노드들의 (x, y) 좌표를 계산!
        # 거리를 넉넉하게 퍼뜨리기 위해 k값을 1.5로 줌
        pos = nx.spring_layout(sub_G, k=1.5, iterations=200, seed=42)

        for node in sub_G.nodes():
            size_node = min(8 + deg[node] * 150, 50)
            border_width = eig_to_border(eig[node])
            
            # 파이썬이 계산한 좌표를 가져와서 1000을 곱해 화면에 널찍하게 뿌려줌
            x_coord = float(pos[node][0]) * 1000
            y_coord = float(pos[node][1]) * 1000

            net.add_node(
                node, label=node,
                x=x_coord, y=y_coord, physics=False, # ✨ 좌표 고정 완료!
                color={
                    "background": color, "border": "#ffffff",
                    "highlight":  {"background": color, "border": "#FFD700"},
                    "hover":      {"background": color, "border": "#FFD700"},
                },
                size=size_node, borderWidth=border_width,
                title=f"키워드: {node}\n대세(Degree): {deg[node]:.4f}\n파워(Eigen): {eig[node]:.4f}"
            )

        if sub_G.number_of_edges() > 0:
            max_w = max(d["weight"] for _, _, d in sub_G.edges(data=True))
            for u, v, data in sub_G.edges(data=True):
                w = data["weight"]
                net.add_edge(
                    u, v, value=w,
                    width=0.5 + 3.0 * (w / max_w),
                    color={"opacity": 0.1 + 0.7 * (w / max_w)},
                    title=f"{u} ↔ {v}\n동시 출현: {w}회"
                )

        safe_name = megatrend.replace("/", "_").replace("·", "_")
        save_path = f"{HTML_SAVE_DIR}comm_{comm_id:02d}_{safe_name}.html"
        net.save_graph(save_path)

        # ✨ 핵심 3: 자바스크립트에서는 '물리엔진 끄기' 로직을 빼고, 클릭 효과만 남김!
        with open(save_path, "r", encoding="utf-8") as f:
            html = f.read()

        custom_js = """
        <script>
        document.addEventListener("DOMContentLoaded", function() {
          // 이미 물리엔진이 꺼져있으므로 안정화(stabilized) 대기 불필요!
          network.on("click", function(params) {
            if (params.nodes.length > 0) {
              var sel = params.nodes[0];
              var conn = network.getConnectedNodes(sel);
              var edges_ = network.getConnectedEdges(sel);
              nodes.update(nodes.get().map(function(n) {
                return { id: n.id, opacity: (n.id === sel || conn.indexOf(n.id) !== -1) ? 1.0 : 0.05 };
              }));
              edges.update(edges.get().map(function(e) {
                return { id: e.id, color: { opacity: edges_.indexOf(e.id) !== -1 ? 0.9 : 0.02 } };
              }));
            } else {
              nodes.update(nodes.get().map(function(n) { return { id: n.id, opacity: 1.0 }; }));
              edges.update(edges.get().map(function(e) { return { id: e.id, color: { opacity: null } }; }));
            }
          });
        });
        </script>
        """
        html = html.replace("</body>", custom_js + "\n</body>")
        with open(save_path, "w", encoding="utf-8") as f:
            f.write(html)

        print(f"  [{comm_id:2d}] {megatrend:20s} | {len(members):4d}개 노드 완료!")

    print(f"\n✅ 시각화 완료! 이제 겹치거나 꿈틀대지 않을 거야!")

if __name__ == "__main__":
    draw_static_networks()

[작업 시작] 춤추지 않는 완전 고정형 네트워크를 그립니다...
  [ 1] 감성 카페 라이프            | 1143개 노드 완료!
  [ 3] 편의점 시즌 콜라보           |  774개 노드 완료!
  [ 7] 이자카야·바 다이닝           |  612개 노드 완료!
  [ 6] 선물·기념일 디저트           |  403개 노드 완료!
  [ 9] 도쿄 미식 여행             |  268개 노드 완료!
  [13] 연말 파인다이닝             |  243개 노드 완료!
  [ 2] 성수·을지로 로컬 맛집         |  233개 노드 완료!
  [14] 여름 보양·청량 푸드          |  186개 노드 완료!
  [ 8] 명절·연휴 간식             |  171개 노드 완료!
  [ 4] 건강식·저속노화             |  114개 노드 완료!
  [11] 미디어 푸드 콘텐츠           |  109개 노드 완료!
  [ 5] 힙한 한끼 미식             |  106개 노드 완료!
  [20] 수험생·이벤트 특수           |  106개 노드 완료!
  [19] 새해·시즌 베이커리           |   79개 노드 완료!
  [10] 로컬 카페 탐방             |   57개 노드 완료!
  [21] 프리미엄 핸드드립·오마카세       |   50개 노드 완료!
  [17] 자극적 한식·스트레스 해소       |   46개 노드 완료!
  [15] 가을·겨울 패션 코디          |   39개 노드 완료!
  [16] 공연·피크닉 아웃도어          |   32개 노드 완료!
  [24] 마곡 로컬 카페             |   10개 노드 완료!
  [25] 인플루언서 맛집             |    7개 노드 완료!
  [22] 이색 요리 실험             |    6개 노드 완료!

✅ 시각화 완료! 이제 겹치거나

In [25]:
import pandas as pd
import json
import ast

# ============================================================
# 1. 파일 경로 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"

# ✨ 네가 알려준 정확한 원본 데이터 경로!
ORIGINAL_CSV_PATH = "/Users/hajiyoon/workspace/data/processed/knewnew_full_2025_v2.csv"
# ✨ 새롭게 컬럼이 추가되어 저장될 새로운 파일 경로 (원본 보호)
FINAL_CSV_PATH    = SAVE_DIR + "knewnew_final_with_v3.csv"

def safe_eval(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [kw.strip() for kw in str(x).strip("[]").split(",") if kw.strip()]

def add_keywords_v3():
    print("[작업 시작] 원본 데이터에 keywords_v3 컬럼을 추가해서 새 파일로 만들게...")

    # 1. 생존 확정 명단(partition.json) 불러오기
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    
    valid_set = set(partition.keys())
    print(f"  ✅ 필터링 기준 단어 수: {len(valid_set)}개")

    # 2. 원본 데이터 불러오기
    df = pd.read_csv(ORIGINAL_CSV_PATH)

    # 3. 필터링 로직
    def get_v3_list(v2_str):
        v2_list = safe_eval(v2_str)
        return [kw for kw in v2_list if kw in valid_set]

    # 4. 새 컬럼 keywords_v3 추가
    df['keywords_v3'] = df['keywords_v2'].apply(get_v3_list)

    # 5. 새로운 파일로 저장 (덮어쓰기 아님!)
    df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"  ✅ 작업 완료! 원본은 안전하게 두고 새 파일로 저장했어: {FINAL_CSV_PATH}")

if __name__ == "__main__":
    add_keywords_v3()

[작업 시작] 원본 데이터에 keywords_v3 컬럼을 추가해서 새 파일로 만들게...
  ✅ 필터링 기준 단어 수: 4794개
  ✅ 작업 완료! 원본은 안전하게 두고 새 파일로 저장했어: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/knewnew_final_with_v3.csv


총 노드 5120개

필터링된 노드 326개

총 개수 4794개

In [26]:
import pandas as pd
import networkx as nx
from itertools import combinations
import ast

# ============================================================
# 1. 파일 경로 및 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
FILE_PATH = SAVE_DIR + "knewnew_final_with_v3.csv"
COLUMN_NAME = "keywords_v3"

def safe_eval(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [kw.strip() for kw in str(x).strip("[]").split(",") if kw.strip()]

def calculate_network_stats():
    print("[작업 시작] 네트워크를 만들고 기초 통계를 계산할게...")

    # 1. 데이터 불러오기
    df = pd.read_csv(FILE_PATH)
    df[COLUMN_NAME] = df[COLUMN_NAME].apply(safe_eval)

    # 2. 네트워크 생성
    G = nx.Graph()
    for keywords in df[COLUMN_NAME]:
        # 키워드가 2개 이상일 때만 연결선(Edge) 생성
        if len(keywords) >= 2:
            for u, v in combinations(keywords, 2):
                if G.has_edge(u, v):
                    G[u][v]["weight"] += 1
                else:
                    G.add_edge(u, v, weight=1)

    # 3. 기초 통계 계산
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    # 평균 연결 정도 (단어 1개당 평균적으로 몇 개와 연결되는지)
    if num_nodes > 0:
        avg_degree = sum(dict(G.degree()).values()) / num_nodes
    else:
        avg_degree = 0
        
    # 밀도 (전체 가능한 연결 중 실제 연결된 비율)
    density = nx.density(G)

    # 연결 요소(떨어져 있는 섬 같은 그룹) 개수
    num_components = nx.number_connected_components(G)

    # 4. 결과 출력
    print("\n📊 [네트워크 기초 통계 결과]")
    print(f"  📍 총 노드(단어) 수: {num_nodes:,}개")
    print(f"  🔗 총 엣지(연결선) 수: {num_edges:,}개")
    print(f"  🤝 평균 연결 수(Average Degree): {avg_degree:.2f}개")
    print(f"  🕸️ 네트워크 밀도(Density): {density:.5f}")
    print(f"  🏝️ 연결 요소(독립된 그룹) 수: {num_components:,}개")

if __name__ == "__main__":
    calculate_network_stats()

[작업 시작] 네트워크를 만들고 기초 통계를 계산할게...

📊 [네트워크 기초 통계 결과]
  📍 총 노드(단어) 수: 4,794개
  🔗 총 엣지(연결선) 수: 40,754개
  🤝 평균 연결 수(Average Degree): 17.00개
  🕸️ 네트워크 밀도(Density): 0.00355
  🏝️ 연결 요소(독립된 그룹) 수: 1개


In [27]:
import pickle
import json
import networkx as nx
import pandas as pd

# ============================================================
# 1. 파일 경로 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"

def analyze_community_one_centrality():
    print("[작업 시작] 1번 군집 '감성 카페 라이프'의 중심성 분석을 시작합니다...")

    # 1. 데이터 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 1번 군집 노드 추출 및 서브 그래프 생성
    target_comm_id = 1
    nodes = [n for n, c in partition.items() if c == target_comm_id]
    
    if not nodes:
        print(f"❌ 군집 {target_comm_id}번에 해당하는 노드가 없습니다.")
        return

    sub_G = G.subgraph(nodes).copy()
    print(f"  📍 분석 대상 노드 수: {len(nodes)}개")

    # 3. 중심성 지표 계산
    # 연결 중심성 (얼마나 많은 이웃을 가졌는가)
    deg_cent = nx.degree_centrality(sub_G)
    
    # 고유벡터 중심성 (얼마나 영향력 있는 이웃과 연결되었는가)
    try:
        eig_cent = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
    except nx.PowerIterationFailedConvergence:
        eig_cent = {n: 0.0 for n in sub_G.nodes()}

    # 4. 데이터프레임 생성 및 합산
    analysis_results = []
    for node in nodes:
        d_val = deg_cent[node]
        e_val = eig_cent[node]
        analysis_results.append({
            "키워드": node,
            "연결_중심성": d_val,
            "고유벡터_중심성": e_val,
            "합산_점수": d_val + e_val  # 두 지표의 합
        })

    df = pd.DataFrame(analysis_results)
    
    # 5. 합산 점수 기준 내림차순 정렬
    df = df.sort_values(by="합산_점수", ascending=False)

    # 6. 결과 출력
    print("\n📊 [1번 군집 중심성 상위 20개 결과]")
    print(df.head(20).to_string(index=False))
    
    # CSV로 저장하고 싶다면 아래 주석을 해제하세요.
    # df.to_csv(SAVE_DIR + "community_1_ranking.csv", index=False, encoding="utf-8-sig")

if __name__ == "__main__":
    analyze_community_one_centrality()

[작업 시작] 1번 군집 '감성 카페 라이프'의 중심성 분석을 시작합니다...
  📍 분석 대상 노드 수: 1143개

📊 [1번 군집 중심성 상위 20개 결과]
  키워드   연결_중심성  고유벡터_중심성    합산_점수
   카페 0.680385  0.546936 1.227322
  디저트 0.320490  0.328507 0.648997
   맛집 0.265324  0.290468 0.555792
   커피 0.268827  0.280268 0.549095
  F&B 0.253940  0.284010 0.537950
  브런치 0.259194  0.263045 0.522240
   여행 0.204028  0.216273 0.420301
  데이트 0.183888  0.166800 0.350688
   힐링 0.182137  0.146619 0.328755
   감성 0.187391  0.140157 0.327547
  빈티지 0.143608  0.138261 0.281869
   여유 0.147110  0.107767 0.254877
 베이커리 0.119089  0.088745 0.207835
   낭만 0.112960  0.072991 0.185951
   신상 0.079685  0.106103 0.185788
   주말 0.098949  0.066420 0.165370
   산책 0.086690  0.078570 0.165260
 당일치기 0.081436  0.059225 0.140661
트렌드세터 0.054291  0.086162 0.140453
   로컬 0.076182  0.059340 0.135523


In [28]:
import pickle
import json
import networkx as nx
import pandas as pd
import os

# ============================================================
# 1. 파일 경로 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"
OUTPUT_PATH    = SAVE_DIR + "community_1_centrality_full_with_avg.csv"

def save_community_one_centrality_with_avg():
    print("[작업 시작] 1번 군집의 중심성과 평균값을 CSV로 저장할게...")

    # 1. 데이터 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 1번 군집 노드 추출 및 서브 그래프 생성
    target_comm_id = 1
    nodes = [n for n, c in partition.items() if c == target_comm_id]
    
    if not nodes:
        print(f"❌ 군집 {target_comm_id}번에 해당하는 노드가 없어.")
        return

    sub_G = G.subgraph(nodes).copy()

    # 3. 중심성 지표 계산
    deg_cent = nx.degree_centrality(sub_G)
    try:
        eig_cent = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
    except nx.PowerIterationFailedConvergence:
        eig_cent = {n: 0.0 for n in sub_G.nodes()}

    # 4. 전체 데이터 수집
    all_results = []
    for node in nodes:
        d_val = deg_cent[node]
        e_val = eig_cent[node]
        all_results.append({
            "키워드": node,
            "연결_중심성": d_val,
            "고유벡터_중심성": e_val,
            "합산_점수": d_val + e_val
        })

    # 5. 데이터프레임 생성 및 내림차순 정렬
    df = pd.DataFrame(all_results)
    df = df.sort_values(by="합산_점수", ascending=False)

    # ✨ 6. 각 지표별 평균 계산 및 추가
    avg_row = pd.DataFrame([{
        "키워드": "[전체 평균]",
        "연결_중심성": df["연결_중심성"].mean(),
        "고유벡터_중심성": df["고유벡터_중심성"].mean(),
        "합산_점수": df["합산_점수"].mean()
    }])
    
    # 정렬된 기존 데이터 아래에 평균값 행을 찰싹 붙이기
    df = pd.concat([df, avg_row], ignore_index=True)

    # 7. CSV 파일로 저장
    df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    
    print(f"✅ 평균 포함 저장 완료! 총 {len(df)-1}개 노드 + 1개 평균 행.")
    print(f"📍 파일 경로: {OUTPUT_PATH}")

if __name__ == "__main__":
    save_community_one_centrality_with_avg()

[작업 시작] 1번 군집의 중심성과 평균값을 CSV로 저장할게...
✅ 평균 포함 저장 완료! 총 1143개 노드 + 1개 평균 행.
📍 파일 경로: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/community_1_centrality_full_with_avg.csv


In [29]:
import pickle
import json
import networkx as nx

# ============================================================
# 1. 파일 경로 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
G_PATH         = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"

def print_top_percentile_keywords():
    print("[작업 시작] 1번 군집의 상위 퍼센트별 키워드를 출력할게...\n")

    # 1. 데이터 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 1번 군집 노드 추출
    target_comm_id = 1
    nodes = [n for n, c in partition.items() if c == target_comm_id]
    
    if not nodes:
        print(f"❌ 군집 {target_comm_id}번에 해당하는 노드가 없어.")
        return

    sub_G = G.subgraph(nodes).copy()

    # 3. 중심성 계산
    deg_cent = nx.degree_centrality(sub_G)
    try:
        eig_cent = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
    except nx.PowerIterationFailedConvergence:
        eig_cent = {n: 0.0 for n in sub_G.nodes()}

    # 4. 합산 점수 계산 및 정렬
    scored_nodes = []
    for node in nodes:
        score = deg_cent[node] + eig_cent[node]
        scored_nodes.append((node, score))
    
    # 점수가 높은 순으로 내림차순 정렬
    scored_nodes.sort(key=lambda x: x[1], reverse=True)

    total_nodes = len(scored_nodes)
    print(f"📍 1번 군집 총 노드 수: {total_nodes}개\n")

    # 5. 퍼센트별 구간 잘라서 출력
    percentiles = [10, 20, 30, 40, 50]
    start_idx = 0

    for p in percentiles:
        # 어디까지 자를지 계산
        end_idx = int(total_nodes * (p / 100))
        
        # 해당 구간의 단어들만 추출
        bracket_nodes = [item[0] for item in scored_nodes[start_idx:end_idx]]
        
        if p == 10:
            label = "상위 0% ~ 10%"
        else:
            label = f"상위 {p-10}% ~ {p}%"
            
        print(f"🟢 {label} ({len(bracket_nodes)}개)")
        print(", ".join(bracket_nodes))
        print("-" * 60)
        
        # 다음 구간을 위해 시작점 업데이트
        start_idx = end_idx

if __name__ == "__main__":
    print_top_percentile_keywords()

[작업 시작] 1번 군집의 상위 퍼센트별 키워드를 출력할게...

📍 1번 군집 총 노드 수: 1143개

🟢 상위 0% ~ 10% (114개)
카페, 디저트, 맛집, 커피, F&B, 브런치, 여행, 데이트, 힐링, 감성, 빈티지, 여유, 베이커리, 낭만, 신상, 주말, 산책, 당일치기, 트렌드세터, 로컬, 아지트, 가오픈, 소품샵, 술, 봄, 아늑함, 분위기, 빵, 샌드위치, 전주, 가을, 드라이브, 카페투어, 망원동, 서촌, 바, 골목, 부산, 로스터리, 해방촌, 베이글, 청춘, 테라스, 소금빵, 술집, 칵테일, 와인바, 휴식, 식사, 뷰, 성북천, 스폿, 이태원, 독서, LP바, 트렌디, LP, 연남동, 양식, 홍대, 오션뷰, 혼자, 트렌드, 대전, 행궁동, 대구, 작업, 책, 고즈넉함, 바다, 젤라또, 한옥, 당충전, 전포, 무드, 구움과자, 경주, 한남동, 통창, 뚜벅이, 커피숍, 인테리어, 청주, 사랑방, 아늑, 울산, 노을, 느좋, 가구, 빵지순례, 이국적, 북촌, 나들이, 드립커피, 연희동, 강릉, 밥, 광주, 외출, 바다뷰, 공간, 군산, 광안리, 밤, 찻집, 다이닝, 먹거리, 합정, 속초, 효창공원, 서울숲, 동네, 기분전환, 서점
------------------------------------------------------------
🟢 상위 10% ~ 20% (114개)
계절, 초여름, 초록, 혼놀, 코지, 벚꽃, 아기자기, 부평, 문래동, 하우스, 시그니처, 제주도, 여유로움, 로스터스, 자연, 국내여행, 대화, 공원, 노트북, 야외, 코스, 포항, 한적함, 공부, 브루잉, 편집숍, 후암동, 돌담길, 킷사텐, 북카페, 아날로그, 주류, 창원, 축제, 컨셉, 해운대, 용산구, 망원, 독립서점, 플랜테리어, 아기자기함, 서울역, 집중, 햇살, 인생샷, 차, 미니멀, 공항, 키티, 고즈넉, 남영동, 원주, 스페셜티커피, 코펜하겐, 감다살, 내향인, 상수, 원두, 서순라길, 김해, 해외여행, 포카치아, 파주, 펍, 보또비노, 뉴뉴APP, 바

In [30]:
import pandas as pd
import networkx as nx
from itertools import combinations
import ast
import pickle
import os

# ============================================================
# 1. 설정 및 경로
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
# 필터링이 완료된 최신 데이터 파일
INPUT_CSV_PATH = SAVE_DIR + "knewnew_final_with_v3.csv"
# 결과 저장 파일
OUTPUT_CSV_PATH = SAVE_DIR + "total_keyword_centrality_results.csv"

def safe_eval(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [kw.strip() for kw in str(x).strip("[]").split(",") if kw.strip()]

def analyze_total_network_centrality():
    print("[작업 시작] 전체 키워드 통합 네트워크 분석을 시작합니다...")

    # 1. 데이터 불러오기
    df = pd.read_csv(INPUT_CSV_PATH)
    df['keywords_v3'] = df['keywords_v3'].apply(safe_eval)

    # 2. 통합 네트워크(G) 생성
    G = nx.Graph()
    for keywords in df['keywords_v3']:
        if len(keywords) >= 2:
            # 한 게시물 내 키워드들의 모든 조합을 연결
            for u, v in combinations(keywords, 2):
                if G.has_edge(u, v):
                    G[u][v]["weight"] += 1
                else:
                    G.add_edge(u, v, weight=1)

    print(f"  📍 생성된 전체 노드 수: {G.number_of_nodes()}개")
    print(f"  🔗 생성된 전체 엣지 수: {G.number_of_edges()}개")

    # 3. 중심성 지표 계산
    print("  중심성 지표 계산 중 (노드가 많으면 시간이 조금 걸릴 수 있어요)...")
    
    # 연결 중심성
    deg_cent = nx.degree_centrality(G)
    
    # 고유벡터 중심성 (가중치 반영)
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight="weight")
    except nx.PowerIterationFailedConvergence:
        print("  ⚠️ 고유벡터 중심성 수렴 실패 - 기본값으로 처리합니다.")
        eig_cent = {n: 0.0 for n in G.nodes()}

    # 4. 결과 데이터 정리
    results = []
    for node in G.nodes():
        d_val = deg_cent[node]
        e_val = eig_cent[node]
        results.append({
            "키워드": node,
            "연결_중심성": d_val,
            "고유벡터_중심성": e_val,
            "중심성_합계": d_val + e_val
        })

    # 5. 데이터프레임 생성 및 정렬
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(by="중심성_합계", ascending=False)

    # 6. CSV 저장
    result_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    
    print(f"✅ 분석 완료! 결과가 저장되었습니다: {OUTPUT_CSV_PATH}")
    print("\n[상위 10개 키워드 미리보기]")
    print(result_df.head(10).to_string(index=False))

if __name__ == "__main__":
    analyze_total_network_centrality()

[작업 시작] 전체 키워드 통합 네트워크 분석을 시작합니다...
  📍 생성된 전체 노드 수: 4794개
  🔗 생성된 전체 엣지 수: 40754개
  중심성 지표 계산 중 (노드가 많으면 시간이 조금 걸릴 수 있어요)...
✅ 분석 완료! 결과가 저장되었습니다: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/total_keyword_centrality_results.csv

[상위 10개 키워드 미리보기]
키워드   연결_중심성  고유벡터_중심성   중심성_합계
 카페 0.253286  0.526751 0.780037
디저트 0.196954  0.331787 0.528741
F&B 0.142917  0.278417 0.421333
 맛집 0.122262  0.275499 0.397760
 커피 0.111412  0.266728 0.378140
브런치 0.109117  0.253776 0.362894
 여행 0.094721  0.204599 0.299321
데이트 0.121636  0.170938 0.292574
 감성 0.078030  0.136290 0.214320
 힐링 0.071771  0.138416 0.210187


In [31]:
import pandas as pd
import pickle
import json

# ============================================================
# 1. 파일 경로 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
CSV_PATH = SAVE_DIR + "total_keyword_centrality_results.csv"
G_PATH = SAVE_DIR + "뉴뉴_network_G.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"

# 원본 보호를 위해 새로 저장할 경로
NEW_G_PATH = SAVE_DIR + "뉴뉴_network_G_filtered.pkl"
NEW_PARTITION_PATH = SAVE_DIR + "뉴뉴_partition_filtered.json"

def filter_network_by_csv():
    print("[작업 시작] CSV 기준에 맞춰 네트워크와 군집을 정리할게...")

    # 1. CSV에서 살릴 키워드 목록 불러오기
    df = pd.read_csv(CSV_PATH)
    
    # 💡 만약 0인 애들을 살리고 싶다면 아래 코드에서 == 1 을 == 0 으로 바꿔줘!
    keep_keywords = set(df[df['살리기_여부'] == 1]['키워드'])
    print(f"  ✅ 살리기로 한 키워드 수: {len(keep_keywords)}개")

    # 2. 기존 네트워크(G)와 군집(partition) 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
        
    # 3. 네트워크에서 지울 노드 찾기
    # 현재 네트워크에 있는 노드 중, keep_keywords 명단에 없는 애들
    nodes_to_remove = [node for node in list(G.nodes) if node not in keep_keywords]
    print(f"  🗑️ 네트워크에서 지워질 노드 수: {len(nodes_to_remove)}개")

    # 4. G와 partition에서 해당 노드들 완전 삭제
    G.remove_nodes_from(nodes_to_remove)
    
    for node in nodes_to_remove:
        if node in partition:
            del partition[node]

    # 5. 정리된 새로운 네트워크와 군집 정보 저장
    with open(NEW_G_PATH, "wb") as f:
        pickle.dump(G, f)
    with open(NEW_PARTITION_PATH, "w", encoding="utf-8") as f:
        json.dump(partition, f, ensure_ascii=False)

    print(f"✅ 작업 완료! 안전하게 새 파일로 저장했어.")
    print(f"  - 저장된 네트워크 노드 수: {G.number_of_nodes()}개")

if __name__ == "__main__":
    filter_network_by_csv()

[작업 시작] CSV 기준에 맞춰 네트워크와 군집을 정리할게...
  ✅ 살리기로 한 키워드 수: 1444개
  🗑️ 네트워크에서 지워질 노드 수: 3350개
✅ 작업 완료! 안전하게 새 파일로 저장했어.
  - 저장된 네트워크 노드 수: 1444개


In [32]:
import os
import pickle
import json
import networkx as nx
from pyvis.network import Network
from collections import Counter

# ============================================================
# 1. 파일 경로 및 설정 
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
# ✨ 아까 만든 필터링된 파일(filtered)을 불러오도록 경로 수정!
G_PATH         = SAVE_DIR + "뉴뉴_network_G_filtered.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition_filtered.json"
# ✨ 결과물이 섞이지 않게 저장 폴더 이름도 새롭게 지정했어!
HTML_SAVE_DIR  = SAVE_DIR + "찐최종!!!community_networks_filtered/"

os.makedirs(HTML_SAVE_DIR, exist_ok=True)

COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 11: "미디어 푸드 콘텐츠", 5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수", 19: "새해·시즌 베이커리", 10: "로컬 카페 탐방", 
    21: "프리미엄 핸드드립·오마카세", 17: "자극적 한식·스트레스 해소",
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
}

PALETTE = [
    "#FF5A5F", "#087E8B", "#3C3C3C", "#F58231", "#911EB4", 
    "#46F0F0", "#F032E6", "#BCF60C", "#FABED4", "#008080",
    "#E6194B", "#3CB44B", "#FFE119", "#4363D8", "#F58231"
]

def draw_static_networks_filtered():
    print("[작업 시작] 필터링된 데이터로 춤추지 않는 고정형 네트워크를 그립니다...")

    # 1. 필터링된 데이터 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    community_sizes = Counter(partition.values())

    # 2. 군집별로 순회하며 시각화
    for comm_id, size in sorted(community_sizes.items(), key=lambda x: x[1], reverse=True):
        if comm_id not in COMMUNITY_NAMES:
            continue
            
        megatrend = COMMUNITY_NAMES.get(comm_id, f"군집_{comm_id}")
        members   = [node for node, c in partition.items() if c == comm_id]
        
        # 만약 이 군집의 노드가 다 필터링돼서 0개가 되었다면 건너뛰기!
        if not members:
            continue
            
        sub_G = G.subgraph(members).copy()

        # 중심성 계산
        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}

        eig_vals = list(eig.values())
        eig_min, eig_max = min(eig_vals) if eig_vals else 0, max(eig_vals) if eig_vals else 0

        def eig_to_border(val):
            return 1 + ((val - eig_min) / (eig_max - eig_min + 1e-9)) * 9

        color = PALETTE[list(sorted(community_sizes, key=community_sizes.get, reverse=True)).index(comm_id) % len(PALETTE)]

        net = Network(height="800px", width="100%", bgcolor="#1a1a2e", font_color="#ffffff", notebook=False)
        
        # 물리 엔진 끄기 (enabled: false)
        net.set_options("""
        {
          "physics": { "enabled": false },
          "nodes": {
            "shape": "dot",
            "font": {"size": 13, "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"},
            "borderWidthSelected": 4
          },
          "edges": {"smooth": {"enabled": true, "type": "continuous"}},
          "interaction": {
            "hover": true, "tooltipDelay": 100, "hideEdgesOnDrag": true,
            "navigationButtons": true, "keyboard": true
          }
        }
        """)

        # 파이썬(NetworkX)을 이용해 미리 (x, y) 좌표 계산
        pos = nx.spring_layout(sub_G, k=1.5, iterations=200, seed=42)

        for node in sub_G.nodes():
            size_node = min(8 + deg[node] * 150, 50)
            border_width = eig_to_border(eig[node])
            
            x_coord = float(pos[node][0]) * 1000
            y_coord = float(pos[node][1]) * 1000

            net.add_node(
                node, label=node,
                x=x_coord, y=y_coord, physics=False, # 좌표 고정
                color={
                    "background": color, "border": "#ffffff",
                    "highlight":  {"background": color, "border": "#FFD700"},
                    "hover":      {"background": color, "border": "#FFD700"},
                },
                size=size_node, borderWidth=border_width,
                title=f"키워드: {node}\n대세(Degree): {deg[node]:.4f}\n파워(Eigen): {eig[node]:.4f}"
            )

        if sub_G.number_of_edges() > 0:
            max_w = max(d["weight"] for _, _, d in sub_G.edges(data=True))
            for u, v, data in sub_G.edges(data=True):
                w = data["weight"]
                net.add_edge(
                    u, v, value=w,
                    width=0.5 + 3.0 * (w / max_w),
                    color={"opacity": 0.1 + 0.7 * (w / max_w)},
                    title=f"{u} ↔ {v}\n동시 출현: {w}회"
                )

        safe_name = megatrend.replace("/", "_").replace("·", "_")
        save_path = f"{HTML_SAVE_DIR}comm_{comm_id:02d}_{safe_name}.html"
        net.save_graph(save_path)

        # 클릭 효과만 남기는 자바스크립트 주입
        with open(save_path, "r", encoding="utf-8") as f:
            html = f.read()

        custom_js = """
        <script>
        document.addEventListener("DOMContentLoaded", function() {
          network.on("click", function(params) {
            if (params.nodes.length > 0) {
              var sel = params.nodes[0];
              var conn = network.getConnectedNodes(sel);
              var edges_ = network.getConnectedEdges(sel);
              nodes.update(nodes.get().map(function(n) {
                return { id: n.id, opacity: (n.id === sel || conn.indexOf(n.id) !== -1) ? 1.0 : 0.05 };
              }));
              edges.update(edges.get().map(function(e) {
                return { id: e.id, color: { opacity: edges_.indexOf(e.id) !== -1 ? 0.9 : 0.02 } };
              }));
            } else {
              nodes.update(nodes.get().map(function(n) { return { id: n.id, opacity: 1.0 }; }));
              edges.update(edges.get().map(function(e) { return { id: e.id, color: { opacity: null } }; }));
            }
          });
        });
        </script>
        """
        html = html.replace("</body>", custom_js + "\n</body>")
        with open(save_path, "w", encoding="utf-8") as f:
            f.write(html)

        print(f"  [{comm_id:2d}] {megatrend:20s} | {len(members):4d}개 노드 완료!")

    print(f"\n✅ 시각화 완료! '{HTML_SAVE_DIR}' 폴더에 예쁘게 저장됐어!")

if __name__ == "__main__":
    draw_static_networks_filtered()

[작업 시작] 필터링된 데이터로 춤추지 않는 고정형 네트워크를 그립니다...
  [ 3] 편의점 시즌 콜라보           |  450개 노드 완료!
  [ 7] 이자카야·바 다이닝           |  263개 노드 완료!
  [ 1] 감성 카페 라이프            |  111개 노드 완료!
  [ 6] 선물·기념일 디저트           |   87개 노드 완료!
  [ 2] 성수·을지로 로컬 맛집         |   79개 노드 완료!
  [14] 여름 보양·청량 푸드          |   66개 노드 완료!
  [ 9] 도쿄 미식 여행             |   65개 노드 완료!
  [13] 연말 파인다이닝             |   63개 노드 완료!
  [ 8] 명절·연휴 간식             |   51개 노드 완료!
  [11] 미디어 푸드 콘텐츠           |   51개 노드 완료!
  [ 4] 건강식·저속노화             |   50개 노드 완료!
  [ 5] 힙한 한끼 미식             |   24개 노드 완료!
  [20] 수험생·이벤트 특수           |   24개 노드 완료!
  [19] 새해·시즌 베이커리           |   18개 노드 완료!
  [17] 자극적 한식·스트레스 해소       |   17개 노드 완료!
  [10] 로컬 카페 탐방             |   10개 노드 완료!
  [15] 가을·겨울 패션 코디          |    5개 노드 완료!
  [21] 프리미엄 핸드드립·오마카세       |    5개 노드 완료!
  [22] 이색 요리 실험             |    3개 노드 완료!
  [25] 인플루언서 맛집             |    2개 노드 완료!

✅ 시각화 완료! '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/찐최종!!!community_networks_filter

In [34]:
import pandas as pd

# 1. 파일 불러오기
file_path = '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords_ip포함.csv'
df = pd.read_csv(file_path)

# 2. 삭제할 문자열 리스트
target_column = 'IP/브랜드/캐릭터/인플루언서 키워드'
tags_to_remove = ['[IP] ', '[브랜드] ', '[유튜버/인플루언서] ', '[캐릭터] ']

# 3. 반복문을 통해 문자열 치환 (삭제)
for tag in tags_to_remove:
    df[target_column] = df[target_column].str.replace(tag, '', regex=False)

# 4. 결과 저장 (원본과 다른 이름으로 저장하는 걸 추천해!)
output_path = '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords_ip포함_v2.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"작업 완료! 파일이 저장되었어: {output_path} 🚀")

작업 완료! 파일이 저장되었어: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords_ip포함_v2.csv 🚀


In [43]:
# ip 브랜드 키워드도 필터링된 노드들 삭제하기
import pandas as pd
import re

# 1. 파일 경로 설정
target_path = '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords_ip포함.csv'
ref_path = '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/total_keyword_centrality_results.csv'
save_path = '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords_ip포함_v2.csv'

# 2. 데이터 불러오기
df_target = pd.read_csv(target_path)
df_ref = pd.read_csv(ref_path)

# 3. 살릴 키워드 목록 추출 (💡 중심성 파일의 키워드 컬럼명이 '키워드'라고 가정했어! 다르면 꼭 수정해줘)
# 검색 속도를 높이기 위해 set 자료형을 사용했어
keep_keywords = set(df_ref[df_ref['살리기_여부'] == 1]['키워드'])

# 4. 키워드 필터링 함수 정의
def filter_words(text):
    if pd.isna(text):
        return text

    result_lines = []
    
    # 엔터(\n)로 구분된 카테고리 줄마다 따로 처리해
    for line in str(text).split('\n'):
        line = line.strip()
        if not line:
            continue

        # '[IP]', '[캐릭터]' 같은 대괄호 태그와 뒤의 단어들을 분리해
        match = re.match(r'(\[.*?\]\s*)(.*)', line)
        if match:
            prefix = match.group(1)
            words_str = match.group(2)
        else:
            prefix = ""
            words_str = line

        # 쉼표(,)로 키워드를 쪼개고, 살릴 키워드 목록(keep_keywords)에 있는 것만 남겨!
        words = [w.strip() for w in words_str.split(',')]
        filtered = [w for w in words if w in keep_keywords]

        # 살아남은 키워드가 하나라도 있으면 원래 태그랑 다시 합쳐줘
        if filtered:
            result_lines.append(prefix + ', '.join(filtered))

    return '\n'.join(result_lines)

# 5. 두 타겟 컬럼에 필터링 함수 적용
cols_to_filter = ['전체_키워드', 'IP/브랜드/캐릭터/인플루언서 키워드']
for col in cols_to_filter:
    df_target[col] = df_target[col].apply(filter_words)

# 6. 새로운 csv 파일로 저장
df_target.to_csv(save_path, index=False, encoding='utf-8-sig')
print("✨ 필터링 완료! 새로운 파일이 저장되었어.")

✨ 필터링 완료! 새로운 파일이 저장되었어.


In [44]:
import pandas as pd

# 1. 파일 경로 설정
file_path = '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_all_keywords_ip포함_v2.csv'

# 2. 데이터 불러오기
df = pd.read_csv(file_path)

# 3. 키워드 개수 업데이트
# 값이 비어있으면 0을 반환하고, 값이 있으면 쉼표(,)로 쪼개서 리스트의 길이를 반환해!
df['키워드_개수'] = df['전체_키워드'].apply(
    lambda x: 0 if pd.isna(x) or str(x).strip() == '' else len(str(x).split(','))
)

# 4. 동일한 파일명으로 덮어써서 저장 (한글 깨짐 방지)
df.to_csv(file_path, index=False, encoding='utf-8-sig')

print("✨ 키워드 개수 업데이트 완료!")

✨ 키워드 개수 업데이트 완료!


In [45]:
import pandas as pd
import pickle
import json
import networkx as nx
import ast

# ============================================================
# 1. 파일 경로 및 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
CSV_PATH = SAVE_DIR + "뉴뉴_community_all_keywords_ip포함.csv"
G_PATH = SAVE_DIR + "뉴뉴_network_G_filtered.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition_filtered.json"

# 결과를 저장할 새로운 파일 경로
OUTPUT_CSV_PATH = SAVE_DIR + "뉴뉴_community_top10_added.csv"

# ✨ 네 CSV 파일에 맞게 아래 두 컬럼 이름을 꼭 확인하고 수정해 줘!
COMMUNITY_COL_NAME = "군집ID"     # (예: '군집ID', 'community', '군집' 등)
KEYWORD_COL_NAME   = "전체_키워드"   # (예: 'keywords', '키워드_v3' 등)

def safe_eval(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [kw.strip() for kw in str(x).strip("[]").split(",") if kw.strip()]

def extract_top_10_percent_keywords():
    print("[작업 시작] 지정된 파일의 키워드로만 네트워크를 그리고 상위 10%를 뽑아낼게! 🚀")

    # 1. 데이터 불러오기
    df = pd.read_csv(CSV_PATH)
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. CSV 파일에서 사용할 키워드 목록만 추출
    csv_keywords = set()
    for kws in df[KEYWORD_COL_NAME].apply(safe_eval):
        csv_keywords.update(kws)
        
    # 기존 파티션에 존재하는 유효한 키워드만 남김
    keep_keywords = set(partition.keys()).intersection(csv_keywords)
    print(f"  ✅ CSV에서 확인된 유효 키워드 수: {len(keep_keywords)}개")

    # 3. 네트워크 필터링 (명단에 없는 단어는 모두 삭제)
    nodes_to_remove = [n for n in G.nodes if n not in keep_keywords]
    G.remove_nodes_from(nodes_to_remove)

    # 4. 군집별 중심성 3종 계산 및 상위 10% 추출
    top10_dict = {} # {군집ID: [상위 10% 키워드 리스트]}
    
    communities = set(partition.values())
    for comm_id in communities:
        # 현재 군집 노드 추출
        nodes = [n for n in keep_keywords if partition.get(n) == comm_id]
        if not nodes:
            continue
            
        sub_G = G.subgraph(nodes).copy()
        
        # 중심성 3대장 계산
        deg_cent = nx.degree_centrality(sub_G)             # 연결 중심성
        bet_cent = nx.betweenness_centrality(sub_G)        # 매개 중심성
        try:
            eig_cent = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight") # 고유벡터 중심성
        except nx.PowerIterationFailedConvergence:
            eig_cent = {n: 0.0 for n in sub_G.nodes()}
            
        # 3가지 점수 합산 ($Score = C_D + C_E + C_B$)
        scores = {}
        for n in nodes:
            scores[n] = deg_cent[n] + eig_cent[n] + bet_cent[n]
            
        # 점수가 높은 순서대로 내림차순 정렬
        sorted_nodes = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)
        
        # 상위 10% 개수 계산 (단어가 너무 적어도 최소 1개는 뽑도록 보장)
        top_k = max(1, int(len(sorted_nodes) * 0.1))
        top10_dict[comm_id] = sorted_nodes[:top_k]
        
        print(f"  - {comm_id}번 군집: 총 {len(nodes)}개 중 상위 {top_k}개 추출 완료!")

    # 5. 원래 데이터프레임에 새로운 컬럼으로 리스트 추가!
    # df의 군집 번호에 맞춰서 매핑해 줌
    df['상위_10퍼센트_핵심키워드'] = df[COMMUNITY_COL_NAME].map(top10_dict)

    # 6. 새로운 파일로 저장
    df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"\n✅ 작업 완료! 리스트가 추가된 새 파일이 저장됐어: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    extract_top_10_percent_keywords()

[작업 시작] 지정된 파일의 키워드로만 네트워크를 그리고 상위 10%를 뽑아낼게! 🚀
  ✅ CSV에서 확인된 유효 키워드 수: 1429개
  - 1번 군집: 총 111개 중 상위 11개 추출 완료!
  - 2번 군집: 총 79개 중 상위 7개 추출 완료!
  - 3번 군집: 총 450개 중 상위 45개 추출 완료!
  - 4번 군집: 총 50개 중 상위 5개 추출 완료!
  - 5번 군집: 총 24개 중 상위 2개 추출 완료!
  - 6번 군집: 총 87개 중 상위 8개 추출 완료!
  - 7번 군집: 총 263개 중 상위 26개 추출 완료!
  - 8번 군집: 총 51개 중 상위 5개 추출 완료!
  - 9번 군집: 총 65개 중 상위 6개 추출 완료!
  - 10번 군집: 총 10개 중 상위 1개 추출 완료!
  - 11번 군집: 총 51개 중 상위 5개 추출 완료!
  - 13번 군집: 총 63개 중 상위 6개 추출 완료!
  - 14번 군집: 총 66개 중 상위 6개 추출 완료!
  - 17번 군집: 총 17개 중 상위 1개 추출 완료!
  - 19번 군집: 총 18개 중 상위 1개 추출 완료!
  - 20번 군집: 총 24개 중 상위 2개 추출 완료!

✅ 작업 완료! 리스트가 추가된 새 파일이 저장됐어: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_top10_added.csv


In [46]:
import pandas as pd
import pickle
import json
import networkx as nx
import ast

# ============================================================
# 1. 파일 경로 및 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
CSV_PATH = SAVE_DIR + "뉴뉴_community_all_keywords_ip포함.csv"
G_PATH = SAVE_DIR + "뉴뉴_network_G_filtered.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition_filtered.json"
OUTPUT_CSV_PATH = SAVE_DIR + "뉴뉴_community_top10_added.csv"


# ✨ 컬럼 이름 설정
COMMUNITY_COL_NAME = "군집ID"
KEYWORD_COL_NAME   = "전체_키워드"
EXCLUDE_COL_NAME   = "IP/브랜드/캐릭터/인플루언서 키워드" # 👈 새로 추가된 제외 기준 컬럼!

def safe_eval(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [kw.strip() for kw in str(x).strip("[]").split(",") if kw.strip()]

def extract_top_10_percent_no_ip():
    print("[작업 시작] 브랜드/IP는 제외하고 순수 일반 키워드만 상위 10%를 뽑아낼게! 🚀")

    # 1. 데이터 불러오기
    df = pd.read_csv(CSV_PATH)
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 사용할 키워드와 '제외할 키워드' 목록 분리 추출
    csv_keywords = set()
    for kws in df[KEYWORD_COL_NAME].apply(safe_eval):
        csv_keywords.update(kws)
        
    keep_keywords = set(partition.keys()).intersection(csv_keywords)
    
    # ✨ 제외 명단(블랙리스트) 만들기
    exclude_keywords = set()
    if EXCLUDE_COL_NAME in df.columns:
        for kws in df[EXCLUDE_COL_NAME].apply(safe_eval):
            exclude_keywords.update(kws)
        print(f"  🚫 제외할 브랜드/IP 키워드 수: {len(exclude_keywords)}개")
    else:
        print(f"  ⚠️ '{EXCLUDE_COL_NAME}' 컬럼이 없어 제외 없이 진행합니다.")

    # 3. 네트워크 뼈대 다듬기 (전체 유효 단어로만)
    nodes_to_remove = [n for n in G.nodes if n not in keep_keywords]
    G.remove_nodes_from(nodes_to_remove)

    # 4. 군집별 중심성 3종 계산 및 필터링
    top10_dict = {}
    communities = set(partition.values())
    
    for comm_id in communities:
        nodes = [n for n in keep_keywords if partition.get(n) == comm_id]
        if not nodes:
            continue
            
        sub_G = G.subgraph(nodes).copy()
        
        # 중심성 계산 (네트워크 형태 유지를 위해 제외 단어도 포함해서 계산해!)
        deg_cent = nx.degree_centrality(sub_G)
        bet_cent = nx.betweenness_centrality(sub_G)
        try:
            eig_cent = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig_cent = {n: 0.0 for n in sub_G.nodes()}
            
        # ✨ 합산 및 필터링 (여기서 브랜드/IP 단어는 점수판에서 빼버려!)
        scores = {}
        for n in nodes:
            if n not in exclude_keywords: # 제외 명단에 없어야만 합격!
                scores[n] = deg_cent[n] + eig_cent[n] + bet_cent[n]
            
        # 점수가 높은 순서대로 내림차순 정렬
        sorted_nodes = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)
        
        # 제외하고 남은 순수 단어 개수를 기준으로 10% 계산
        top_k = max(1, int(len(sorted_nodes) * 0.1))
        top10_dict[comm_id] = sorted_nodes[:top_k]
        
        print(f"  - {comm_id}번 군집: (순수 단어 {len(sorted_nodes)}개 중) 상위 {top_k}개 추출 완료!")

    # 5. 리스트 추가 및 저장
    df['상위_10퍼센트_핵심키워드(IP제외)'] = df[COMMUNITY_COL_NAME].map(top10_dict)
    df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"\n✅ 작업 완료! 브랜드가 쏙 빠진 새 파일이 저장됐어: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    extract_top_10_percent_no_ip()

[작업 시작] 브랜드/IP는 제외하고 순수 일반 키워드만 상위 10%를 뽑아낼게! 🚀
  🚫 제외할 브랜드/IP 키워드 수: 198개
  - 1번 군집: (순수 단어 105개 중) 상위 10개 추출 완료!
  - 2번 군집: (순수 단어 76개 중) 상위 7개 추출 완료!
  - 3번 군집: (순수 단어 386개 중) 상위 38개 추출 완료!
  - 4번 군집: (순수 단어 48개 중) 상위 4개 추출 완료!
  - 5번 군집: (순수 단어 18개 중) 상위 1개 추출 완료!
  - 6번 군집: (순수 단어 60개 중) 상위 6개 추출 완료!
  - 7번 군집: (순수 단어 241개 중) 상위 24개 추출 완료!
  - 8번 군집: (순수 단어 44개 중) 상위 4개 추출 완료!
  - 9번 군집: (순수 단어 49개 중) 상위 4개 추출 완료!
  - 10번 군집: (순수 단어 10개 중) 상위 1개 추출 완료!
  - 11번 군집: (순수 단어 41개 중) 상위 4개 추출 완료!
  - 13번 군집: (순수 단어 42개 중) 상위 4개 추출 완료!
  - 14번 군집: (순수 단어 63개 중) 상위 6개 추출 완료!
  - 17번 군집: (순수 단어 17개 중) 상위 1개 추출 완료!
  - 19번 군집: (순수 단어 13개 중) 상위 1개 추출 완료!
  - 20번 군집: (순수 단어 18개 중) 상위 1개 추출 완료!

✅ 작업 완료! 브랜드가 쏙 빠진 새 파일이 저장됐어: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_top10_added.csv


In [50]:
import pandas as pd
import pickle
import ast
import json
import networkx as nx

# ============================================================
# 1. 파일 경로 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
CSV_PATH = SAVE_DIR + "뉴뉴_community_top10_added.csv"
G_PATH = SAVE_DIR + "뉴뉴_network_G_filtered.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"

# ✨ 최종 저장될 파일 경로
OUTPUT_CSV_PATH = SAVE_DIR + "뉴뉴_community_1hop_mapping_final.csv"

TOP10_COL_NAME = "상위_10퍼센트_핵심키워드(IP제외)"
COMMUNITY_COL_NAME = "군집ID" 

def safe_eval(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [kw.strip() for kw in str(x).strip("[]").split(",") if kw.strip()]

def save_1hop_mapping_to_csv():
    print("[작업 시작] 군집 내부 1-hop 매핑 결과를 CSV로 저장할게! 🚀")

    # 1. 데이터, 네트워크, 군집 정보 불러오기
    df = pd.read_csv(CSV_PATH)
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}

    # 2. 각 행(군집)별로 같은 군집 내 1-hop만 매핑하는 함수
    def get_community_1hop(row):
        comm_id = row[COMMUNITY_COL_NAME]
        top10_kws = safe_eval(row[TOP10_COL_NAME])
        mapping_dict = {}
        
        for kw in top10_kws:
            if kw in G:
                # 같은 군집 소속인 이웃만 필터링
                valid_neighbors = [n for n in G.neighbors(kw) if partition.get(n) == comm_id]
                mapping_dict[kw] = valid_neighbors
                
        return mapping_dict

    # 3. 새로운 컬럼 생성
    if TOP10_COL_NAME in df.columns:
        df['노드별_1홉_매핑'] = df.apply(get_community_1hop, axis=1)
    else:
        print(f"  ❌ 에러: '{TOP10_COL_NAME}' 컬럼을 찾을 수 없어.")
        return

    # 4. CSV 파일로 저장 (Excel 호환 인코딩)
    df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    
    print(f"✅ 저장 완료! 파일 경로: {OUTPUT_CSV_PATH}")
    
    # 상위 3개 행만 살짝 보여주기
    print("\n[데이터 저장 예시]")
    print(df[[COMMUNITY_COL_NAME, '노드별_1홉_매핑']].head(3))

if __name__ == "__main__":
    save_1hop_mapping_to_csv()

[작업 시작] 군집 내부 1-hop 매핑 결과를 CSV로 저장할게! 🚀
✅ 저장 완료! 파일 경로: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_community_1hop_mapping_final.csv

[데이터 저장 예시]
   군집ID  \
0     1   
1     3   
2     7   

                                                                                             노드별_1홉_매핑  
0  {'커피': ['빵', '베이커리', '동백꽃필무렵', '칵테일', '당충전', '베이글', '오니기리', '샌드위치', '밥', '풍전솥밥', 'K-시티', '건강한재료'...  
1  {'말차': ['푸딩', '옥수수', '우지', '도라모찌', '라떼', '멜론빵', '멜론', '피스타치오', '제로', 'CU', 'GS25', '유기농', '블랙핑크'...  
2  {'와인': ['한우', '제철식재료', '해산물', '어물전', '송이버섯', '스테이크', '페어링', '밀키트', '알리오올리오', '명란', '파스타', '김태성셰프...  


In [51]:
import pandas as pd
import pickle
import json
import ast

# ============================================================
# 1. 파일 경로 설정
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"
CSV_PATH = SAVE_DIR + "뉴뉴_community_1hop_mapping_final.csv"
G_PATH = SAVE_DIR + "뉴뉴_network_G_filtered.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition.json"

# ✨ 업데이트된 정보를 저장할 새로운 파일 경로
NEW_G_PATH = SAVE_DIR + "뉴뉴_network_G_1hop_only.pkl"
NEW_PARTITION_PATH = SAVE_DIR + "뉴뉴_partition_1hop_only.json"

def keep_only_1hop_nodes():
    print("[작업 시작] 상위 10%와 1홉 이웃만 남기고 나머지는 네트워크에서 삭제할게! 🚀")

    # 1. 데이터 불러오기
    df = pd.read_csv(CSV_PATH)
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)

    # 2. 살려둘 노드(키워드) 목록 수집하기
    keep_nodes = set()
    
    for mapping_str in df['노드별_1홉_매핑']:
        if pd.isna(mapping_str): continue
        
        try:
            mapping_dict = ast.literal_eval(mapping_str)
            for top_node, neighbors in mapping_dict.items():
                keep_nodes.add(top_node)          # 상위 10% 핵심 노드 추가
                keep_nodes.update(neighbors)      # 1홉 이웃 노드들 추가
        except (ValueError, SyntaxError):
            pass
            
    print(f"  ✅ 최종 생존 명단(핵심+1홉) 노드 수: {len(keep_nodes)}개")

    # 3. 네트워크(G)와 군집(partition)에서 불필요한 노드 삭제하기
    nodes_to_remove = [n for n in G.nodes if n not in keep_nodes]
    print(f"  🗑️ 네트워크에서 삭제될 노드 수: {len(nodes_to_remove)}개")
    
    G.remove_nodes_from(nodes_to_remove)
    
    for node in nodes_to_remove:
        if node in partition:
            del partition[node]

    # 4. 업데이트된 정보 저장하기
    with open(NEW_G_PATH, "wb") as f:
        pickle.dump(G, f)
    with open(NEW_PARTITION_PATH, "w", encoding="utf-8") as f:
        json.dump(partition, f, ensure_ascii=False)

    print(f"✅ 업데이트 완료! 불필요한 노드를 삭제하고 새로운 파일로 저장했어.")
    print(f"  - 📍 저장된 네트워크 경로: {NEW_G_PATH}")
    print(f"  - 📍 저장된 군집 정보 경로: {NEW_PARTITION_PATH}")

if __name__ == "__main__":
    keep_only_1hop_nodes()

[작업 시작] 상위 10%와 1홉 이웃만 남기고 나머지는 네트워크에서 삭제할게! 🚀
  ✅ 최종 생존 명단(핵심+1홉) 노드 수: 931개
  🗑️ 네트워크에서 삭제될 노드 수: 513개
✅ 업데이트 완료! 불필요한 노드를 삭제하고 새로운 파일로 저장했어.
  - 📍 저장된 네트워크 경로: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_network_G_1hop_only.pkl
  - 📍 저장된 군집 정보 경로: /Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/뉴뉴_partition_1hop_only.json


In [53]:
import os
import pickle
import json
import networkx as nx
import pandas as pd
import ast
from pyvis.network import Network
from collections import Counter

# ============================================================
# 1. 파일 경로 및 설정 
# ============================================================
SAVE_DIR = "/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/"

G_PATH         = SAVE_DIR + "뉴뉴_network_G_1hop_only.pkl"
PARTITION_PATH = SAVE_DIR + "뉴뉴_partition_1hop_only.json"
CSV_PATH       = SAVE_DIR + "뉴뉴_community_1hop_mapping_final.csv"
HTML_SAVE_DIR  = SAVE_DIR + "1홉만!!community_networks_1hop_only/"

os.makedirs(HTML_SAVE_DIR, exist_ok=True)

TOP10_COL_NAME = "상위_10퍼센트_핵심키워드(IP제외)"

COMMUNITY_NAMES = {
    1: "감성 카페 라이프", 3: "편의점 시즌 콜라보", 7: "이자카야·바 다이닝",
    6: "선물·기념일 디저트", 9: "도쿄 미식 여행", 13: "연말 파인다이닝",
    2: "성수·을지로 로컬 맛집", 14: "여름 보양·청량 푸드", 8: "명절·연휴 간식",
    4: "건강식·저속노화", 11: "미디어 푸드 콘텐츠", 5: "힙한 한끼 미식",
    20: "수험생·이벤트 특수", 19: "새해·시즌 베이커리", 10: "로컬 카페 탐방", 
    21: "프리미엄 핸드드립·오마카세", 17: "자극적 한식·스트레스 해소",
    15: "가을·겨울 패션 코디", 16: "공연·피크닉 아웃도어", 24: "마곡 로컬 카페",
    22: "이색 요리 실험", 12: "빈티지 편집샵", 25: "인플루언서 맛집",
    18: "일본식 선술집"
}

PALETTE = [
    "#FF5A5F", "#087E8B", "#3C3C3C", "#F58231", "#911EB4", 
    "#46F0F0", "#F032E6", "#BCF60C", "#FABED4", "#008080",
    "#E6194B", "#3CB44B", "#FFE119", "#4363D8", "#F58231"
]

def safe_eval(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [kw.strip() for kw in str(x).strip("[]").split(",") if kw.strip()]

def draw_1hop_networks_fixed():
    print("[작업 시작] 에러를 방지하고 핵심 1홉 네트워크 지도를 그릴게! 🚀\n")

    # 1. 데이터 불러오기
    with open(G_PATH, "rb") as f:
        G = pickle.load(f)
    with open(PARTITION_PATH, "r", encoding="utf-8") as f:
        partition = json.load(f)
    partition = {k: int(v) for k, v in partition.items()}
    
    # 상위 10% 핵심 노드 목록 수집 (별 모양을 위해)
    df = pd.read_csv(CSV_PATH)
    top10_nodes_all = set()
    for list_str in df[TOP10_COL_NAME]:
        top10_nodes_all.update(safe_eval(list_str))

    community_sizes = Counter(partition.values())
    sorted_comms = sorted(community_sizes.items(), key=lambda x: x[1], reverse=True)

    # 2. 군집별로 순회하며 시각화
    for comm_id, size in sorted_comms:
        if comm_id not in COMMUNITY_NAMES:
            continue
            
        megatrend = COMMUNITY_NAMES.get(comm_id, f"군집_{comm_id}")
        members   = [node for node, c in partition.items() if c == comm_id]
        
        # 💡 [핵심 에러 방지] 남은 노드가 하나도 없으면 쿨하게 패스!
        if not members:
            print(f"  - [{comm_id:2d}] {megatrend:20s} | 텅 빈 군집이라 건너뛸게!")
            continue
            
        sub_G = G.subgraph(members).copy()

        # 💡 [이중 안전장치] 서브 그래프에 진짜 노드가 있는지 확인
        if sub_G.number_of_nodes() == 0:
            continue

        # 3. 중심성 계산
        deg = nx.degree_centrality(sub_G)
        try:
            eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
        except nx.PowerIterationFailedConvergence:
            eig = {n: 0.0 for n in sub_G.nodes()}

        eig_vals = list(eig.values())
        eig_min, eig_max = min(eig_vals) if eig_vals else 0, max(eig_vals) if eig_vals else 0

        def eig_to_border(val):
            return 1 + ((val - eig_min) / (eig_max - eig_min + 1e-9)) * 9

        color_idx = list(dict(sorted_comms).keys()).index(comm_id) % len(PALETTE)
        color = PALETTE[color_idx]

        # 4. Pyvis 네트워크 설정 (고정형)
        net = Network(height="800px", width="100%", bgcolor="#1a1a2e", font_color="#ffffff", notebook=False)
        net.set_options("""
        {
          "physics": { "enabled": false },
          "nodes": {
            "font": {"size": 13, "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"},
            "borderWidthSelected": 4
          },
          "edges": {"smooth": {"enabled": true, "type": "continuous"}},
          "interaction": {
            "hover": true, "tooltipDelay": 100, "hideEdgesOnDrag": true,
            "navigationButtons": true, "keyboard": true
          }
        }
        """)

        # 좌표 계산
        pos = nx.spring_layout(sub_G, k=1.5, iterations=200, seed=42)

        # 5. 노드 및 엣지 추가
        for node in sub_G.nodes():
            size_node = min(8 + deg[node] * 150, 50)
            border_width = eig_to_border(eig[node])
            x_coord = float(pos[node][0]) * 1000
            y_coord = float(pos[node][1]) * 1000
            
            # 핵심 단어는 별(star), 1홉 이웃은 동그라미(dot)
            shape_node = "star" if node in top10_nodes_all else "dot"

            net.add_node(
                node, label=node,
                x=x_coord, y=y_coord, physics=False, 
                shape=shape_node, 
                color={
                    "background": color, "border": "#ffffff",
                    "highlight":  {"background": color, "border": "#FFD700"},
                    "hover":      {"background": color, "border": "#FFD700"},
                },
                size=size_node, borderWidth=border_width,
                title=f"키워드: {node}\n대세(Degree): {deg[node]:.4f}\n파워(Eigen): {eig[node]:.4f}"
            )

        if sub_G.number_of_edges() > 0:
            max_w = max(d["weight"] for _, _, d in sub_G.edges(data=True))
            for u, v, data in sub_G.edges(data=True):
                w = data["weight"]
                net.add_edge(
                    u, v, value=w,
                    width=0.5 + 3.0 * (w / max_w),
                    color={"opacity": 0.1 + 0.7 * (w / max_w)},
                    title=f"{u} ↔ {v}\n동시 출현: {w}회"
                )

        # 6. 파일 저장 및 자바스크립트 주입
        safe_name = megatrend.replace("/", "_").replace("·", "_")
        save_path = f"{HTML_SAVE_DIR}comm_{comm_id:02d}_{safe_name}.html"
        net.save_graph(save_path)

        with open(save_path, "r", encoding="utf-8") as f:
            html = f.read()

        custom_js = """
        <script>
        document.addEventListener("DOMContentLoaded", function() {
          network.on("click", function(params) {
            if (params.nodes.length > 0) {
              var sel = params.nodes[0];
              var conn = network.getConnectedNodes(sel);
              var edges_ = network.getConnectedEdges(sel);
              nodes.update(nodes.get().map(function(n) {
                return { id: n.id, opacity: (n.id === sel || conn.indexOf(n.id) !== -1) ? 1.0 : 0.05 };
              }));
              edges.update(edges.get().map(function(e) {
                return { id: e.id, color: { opacity: edges_.indexOf(e.id) !== -1 ? 0.9 : 0.02 } };
              }));
            } else {
              nodes.update(nodes.get().map(function(n) { return { id: n.id, opacity: 1.0 }; }));
              edges.update(edges.get().map(function(e) { return { id: e.id, color: { opacity: null } }; }));
            }
          });
        });
        </script>
        """
        html = html.replace("</body>", custom_js + "\n</body>")
        with open(save_path, "w", encoding="utf-8") as f:
            f.write(html)

        print(f"  ✅ [{comm_id:2d}] {megatrend:20s} | {len(members):4d}개 노드 시각화 완료!")

    print(f"\n🎉 작업 끝! '{HTML_SAVE_DIR}' 폴더를 확인해봐!")

if __name__ == "__main__":
    draw_1hop_networks_fixed()

[작업 시작] 에러를 방지하고 핵심 1홉 네트워크 지도를 그릴게! 🚀

  ✅ [ 1] 감성 카페 라이프            | 1094개 노드 시각화 완료!
  ✅ [ 3] 편의점 시즌 콜라보           |  725개 노드 시각화 완료!
  ✅ [ 7] 이자카야·바 다이닝           |  564개 노드 시각화 완료!
  ✅ [ 6] 선물·기념일 디저트           |  345개 노드 시각화 완료!
  ✅ [ 9] 도쿄 미식 여행             |  225개 노드 시각화 완료!
  ✅ [13] 연말 파인다이닝             |  192개 노드 시각화 완료!
  ✅ [ 2] 성수·을지로 로컬 맛집         |  186개 노드 시각화 완료!
  ✅ [14] 여름 보양·청량 푸드          |  156개 노드 시각화 완료!
  ✅ [ 8] 명절·연휴 간식             |  147개 노드 시각화 완료!
  ✅ [ 4] 건강식·저속노화             |   97개 노드 시각화 완료!
  ✅ [20] 수험생·이벤트 특수           |   94개 노드 시각화 완료!
  ✅ [ 5] 힙한 한끼 미식             |   90개 노드 시각화 완료!
  ✅ [11] 미디어 푸드 콘텐츠           |   76개 노드 시각화 완료!
  ✅ [19] 새해·시즌 베이커리           |   69개 노드 시각화 완료!
  ✅ [10] 로컬 카페 탐방             |   53개 노드 시각화 완료!
  ✅ [17] 자극적 한식·스트레스 해소       |   39개 노드 시각화 완료!

🎉 작업 끝! '/Users/hajiyoon/workspace/711project/tests/뉴뉴 네트워크그리기/1홉만!!community_networks_1hop_only/' 폴더를 확인해봐!
